In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from joblib import Parallel, delayed
from datetime import datetime
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import vars

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_colwidth', 200)

color_back = "#0d1117"

path_repo = vars.path_repo
path_data = vars.path_data

In [ ]:
symbs = vars.symbs

symbs_crypto = [x for x in symbs if "-USD" in x]
symbs_forex = [x for x in symbs if "=X" in x]
symbs_stocks = [x for x in symbs if x not in symbs_crypto + symbs_forex]

print("Total:", len(symbs))
print("Crypto :", len(symbs_crypto), f"({(len(symbs_crypto)/len(symbs))*100:.0f}%)")
print("Forex:", len(symbs_forex), f"({(len(symbs_forex)/len(symbs))*100:.0f}%)")
print("Stocks:", len(symbs_stocks), f"({(len(symbs_stocks)/len(symbs))*100:.0f}%)")

In [ ]:
# Process signals in symbol
import yfinance as yf
SYMBOLS = symbs 

# EMAS
EMA_SHORT       = 20
EMA_MID         = 50
EMA_LONG        = 200
# RSI
RSI_PERIOD      = 14
# MACD
MACD_FAST       = 12
MACD_SLOW       = 26
MACD_SIGNAL     = 9
# Bollinger
BB_PERIOD       = 20
BB_STD          = 2.0
# Estocastico
STOCH_K         = 9
STOCH_D         = 3
STOCH_SLOWING   = 3  
# ATR
ATR_PERIOD      = 14
# ATR - Gestión de riesgo
ATR_MULT_SL     = 1.5           # Multiplicador ATR para Stop Loss
ATR_MULT_TP     = 3.0           # Multiplicador ATR para Take Profit (ratio 2:1)
# Parámetro principal: mínimo de reglas que deben cumplirse 
# Rango válido: 1–8. Recomendado: 5 (conservador-moderado), 4 (agresivo), 6 (muy conservador)
MIN_RULES_TO_SIGNAL = 5
# Umbrales RSI 
RSI_OVERSOLD    = 35 # 35
RSI_OVERBOUGHT  = 70 # 65
# Umbrales Estocástico
STOCH_OVERSOLD  = 25
STOCH_OVERBOUGHT = 75
# Umbrales de posición dentro del canal BB
BB_ENTRY_SELL_MIN  = 0.75   # Para VENDER: precio debe estar en el > 75% del canal
BB_ENTRY_BUY_MAX   = 0.25   # Para COMPRAR: precio debe estar en el < 25% del canal
BB_WAIT_SELL_MIN   = 0.45   # Por debajo de esto en venta  → esperar subida
BB_WAIT_BUY_MAX    = 0.55   # Por encima de esto en compra → esperar bajada
# VPVMA
BANDWITH=0.1 # 0.1 filtro para reducir señales falsas (default 0.1)
# Volatility
VOLATILITY_PERIOD = 20

import pandas as pd
def calc_ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=period, adjust=False).mean()


def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calc_macd(close: pd.Series, fast: int, slow: int, signal: int):
    ema_fast   = calc_ema(close, fast)
    ema_slow   = calc_ema(close, slow)
    macd_line  = ema_fast - ema_slow
    signal_line = calc_ema(macd_line, signal)
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram


def calc_bollinger(close: pd.Series, period: int, std_mult: float):
    mid   = close.rolling(period).mean()
    std   = close.rolling(period).std()
    upper = mid + std_mult * std
    lower = mid - std_mult * std
    return upper, mid, lower


def calc_stochastic(high: pd.Series, low: pd.Series, close: pd.Series,
                    k_period: int, slowing: int, d_period: int):
    lowest_low   = low.rolling(k_period).min()
    highest_high = high.rolling(k_period).max()
    denom = (highest_high - lowest_low).replace(0, np.nan)
    
    raw_k = 100 * (close - lowest_low) / denom  # %K crudo
    k     = raw_k.rolling(slowing).mean()        # %K suavizado (Slowing)
    d     = k.rolling(d_period).mean()           # %D
    return k, d


def calc_atr(high: pd.Series, low: pd.Series, close: pd.Series,
             period: int) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, min_periods=period).mean()


def calc_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()


def calc_adx(high, low, close, period=14):
    tr = calc_atr(high, low, close, period) 

    up_move   = high.diff()
    down_move = -low.diff()

    plus_dm  = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)

    plus_dm  = pd.Series(plus_dm,  index=close.index)
    minus_dm = pd.Series(minus_dm, index=close.index)

    atr      = tr
    plus_di  = 100 * plus_dm.ewm(com=period-1).mean()  / atr
    minus_di = 100 * minus_dm.ewm(com=period-1).mean() / atr

    dx  = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    adx = dx.ewm(com=period-1).mean()

    return adx, plus_di, minus_di


def calc_bb_position(df):
    """
    Retorna la posición relativa del precio dentro del canal BB.
      0.0 → precio en la banda inferior
      0.5 → precio en la media (mitad del canal)
      1.0 → precio en la banda superior
    """
    return (df["Close"] - df["BB_lower"]) / (df["BB_upper"] - df["BB_lower"] + 1e-9)


def calculate_vpvma(df, window_fast=12, window_slow=26, window_sign=9):
    """
    VPVMA - Volume Price Volatility Moving Average
    Derivado del MACD por Pat Tong Chio (2022)
    
    Parámetros:
        df           : DataFrame con columnas Open, High, Low, Close, Volume
        window_fast  : período corto (default 12)
        window_slow  : período largo (default 26)
        window_sign  : período de señal (default 9)
        bandwidth    : filtro para reducir señales falsas (default 0.1)
    """
    df["TP"] = (df["High"] + df["Low"] + df["Close"]) / 3

    df["TP_x_Vol"] = df["TP"] * df["Volume"]

    df["SVWMA"] = (
        df["TP_x_Vol"].rolling(window=window_fast).sum() /
        df["Volume"].rolling(window=window_fast).sum()
    )

    df["LVWMA"] = (
        df["TP_x_Vol"].rolling(window=window_slow).sum() /
        df["Volume"].rolling(window=window_slow).sum()
    )

    df["DV"] = df[["Open", "High", "Low", "Close"]].std(axis=1)

    df["SVWMA_x_DV"] = df["SVWMA"] * df["DV"]
    df["LVWMA_x_DV"] = df["LVWMA"] * df["DV"]

    df["ESVMap"] = df["SVWMA_x_DV"].ewm(span=window_fast, adjust=False).mean()
    df["ELVMap"] = df["LVWMA_x_DV"].ewm(span=window_slow, adjust=False).mean()

    df["VPVMA"]  = df["ESVMap"] - df["ELVMap"]
    df["VPVMAS"] = df["VPVMA"].rolling(window=window_sign).mean()
    df.drop(columns = ["SVWMA", "TP", "TP_x_Vol", "SVWMA", "DV", "SVWMA_x_DV", "LVWMA_x_DV", "ESVMap", "ELVMap"], inplace = True)    


    return df


def evaluate_entry_quality(signal: str, bb_pos: float) -> dict:
    """
    Evalúa si el precio está en una zona ÓPTIMA para entrar o si es necesario esperar el REBOTE. 
    Se valida que no esteen la mitad del canal, sino en zonas XTREMs (superior para venta, inferior para compra).
    
    Zonas para VENDER (bb_pos):
    🟢 ZONA ÓPTIMA VENTA    (>= 0.75)      
    🟡 ZONA ACEPTABLE VENTA (0.55 – 0.75)  
    🔴 ESPERAR SUBIDA       (< 0.55)       

    Zonas para COMPRAR (bb_pos):
    🔴 ESPERAR BAJADA       (> 0.55)      
    🟡 ZONA ACEPTABLE COMPRA (0.25 – 0.45) 
    🟢 ZONA ÓPTIMA COMPRA   (<= 0.25) 
    """
    pct = bb_pos * 100 

    if -1 in signal:
        if bb_pos >= BB_ENTRY_SELL_MIN:
            return {
                "entry_ok":   True,
                "advice": -1,
                "entry_zone":     f-1,
                "bb_pct":     pct,
            }
        elif bb_pos >= BB_WAIT_SELL_MIN:
            return {
                "entry_ok":   False,
                "advice": "🟡", # Nivel aceptable, mejor esperar 
                "entry_zone":     (f"🟡 > {BB_ENTRY_SELL_MIN*100:.0f}%"),
                "bb_pct":     pct,
            }
        else:
            return {
                "entry_ok":   False,
                "advice": "⚪",
                "entry_zone":     (f"⚪ > {BB_ENTRY_SELL_MIN*100:.0f}%"), 
                "bb_pct":     pct,
            }

    elif 1 in signal:
        if bb_pos <= BB_ENTRY_BUY_MAX:
            return {
                "entry_ok":   True,
                "advice": 1,
                "entry_zone":  f1,
                "bb_pct":     pct,
            }
        elif bb_pos <= BB_WAIT_BUY_MAX:
            return {
                "entry_ok":   False,
                "advice": "🟡",  
                "entry_zone":     (f"🟡 < {BB_ENTRY_BUY_MAX*100:.0f}%"),
                "bb_pct":     pct,
            }
        else:
            return {
                "entry_ok":   False,
                "advice": "⚪",
                "entry_zone":     (f"⚪ < {BB_ENTRY_BUY_MAX*100:.0f}%"),
                "bb_pct":     pct,
            }

    # Para ESPERAR no aplica
    return {"entry_ok": False, "entry_zone": "", "advice": "", "bb_pct": pct}

def calc_consistencia(close: pd.Series) -> pd.Series:
    prev26 = close.shift(26)
    prev52 = close.shift(52)

    return prev26, prev52

def build_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c, h, l, v = df["Close"], df["High"], df["Low"], df["Volume"]

    df["EMA20"]  = calc_ema(c, EMA_SHORT)
    df["EMA50"]  = calc_ema(c, EMA_MID)
    df["EMA200"] = calc_ema(c, EMA_LONG)

    df["RSI"] = calc_rsi(c, RSI_PERIOD)

    df["MACD"], df["MACD_signal"], df["MACD_hist"] = calc_macd(
        c, MACD_FAST, MACD_SLOW, MACD_SIGNAL
    )

    df["BB_upper"], df["BB_mid"], df["BB_lower"] = calc_bollinger(c, BB_PERIOD, BB_STD)

    df["STOCH_K"], df["STOCH_D"] = calc_stochastic(h, l, c, STOCH_K, STOCH_SLOWING, STOCH_D)

    df["ATR"] = calc_atr(h, l, c, ATR_PERIOD)

    df["OBV"]     = calc_obv(c, v)
    df["OBV_EMA"] = calc_ema(df["OBV"], 20)

    df["ADX"], df["ADX+DI"], df["ADX-DI"] = calc_adx(h, l, c, period=14)

    df["prev26"], df["prev52"] = calc_consistencia(c)

    df = calculate_vpvma(df)

    df["VWAP"] = (df["Close"] * df["Volume"]).cumsum() / df["Volume"].cumsum()

    df["spread"] = (df["High"] - df["Low"]) / df["Close"]
    
    df["vol_ma"] = df["Volume"].rolling(20).mean()
    df["volume_increasing"] = df["Volume"] > df["vol_ma"]
    
    df["order_flow"] = np.where(df["Close"] > df["Open"], "buy", "sell")

    df["volatility"] = df["Close"].pct_change().rolling(VOLATILITY_PERIOD).std()

    # df.dropna(inplace=True)
    return df


def fmt(n, decimals=4) -> str:
    if n is None:
        return "N/A"
    fmt_str = f"{{:,.{decimals}f}}"
    return fmt_str.format(n)


def compute_tp_sl(price: float, atr: float, direction: str) -> tuple:
    """
    Retorna (take_profit, stop_loss) basados en ATR.
    direction: 'BUY' o 'SELL'
    """
    if direction == 1:
        sl = price - ATR_MULT_SL * atr
        tp = price + ATR_MULT_TP * atr
    else:  # SELL
        sl = price + ATR_MULT_SL * atr
        tp = price - ATR_MULT_TP * atr
    return tp, sl

def process_signals(symbol: str, INTERVAL:int, LOOKBACK_DAYS:int) -> dict:
    # symbol = "CVS" # usar antes del 15/04 ya que 3 estrategias se alinean para sell
    # LOOKBACK_DAYS = 40
    # INTERVAL = "1h"

    for _ in range(3):
        try:
            df = yf.download(
                symbol,
                period=f"{LOOKBACK_DAYS}d",
                interval=INTERVAL,
                progress=False,
                auto_adjust=True,
            )
        except Exception as e:
            print(f"ERROR: {symbol} - {e}")
            return pd.DataFrame()

        if len(df) > 0:
            break   

    if df.empty or len(df) < EMA_LONG + 10:
        print(f"ERROR: {symbol} - len(df) < {EMA_LONG + 10}")
        return pd.DataFrame()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    if len(df) < 2:
        print(f"ERROR: {symbol} - len(df) < 2")
        return pd.DataFrame()
    
    df["date"] = df.index
    df = build_indicators(df.copy())
    bb_pos   = calc_bb_position(df)


    # ------------------------------- SEÑALES -------------------------------#
    
    #---------------------------------- VERSIONES MACD + RSI
    # ----------- 1.1 MACD + RSI exigente
    """
    RSI bajo durante varias velas = presión vendedora estructural
    MACD cruzando = inicio del cambio
    Es un detector de “agotamiento + giro temprano”
    """
    df["rsi_below_35"] = (df["RSI"] < 30).astype(int)
    df["rsi_above_65"] = (df["RSI"] > 70).astype(int)

    df["count_men"] = df["rsi_below_35"].rolling(6, min_periods = 1).sum()
    df["count_may"] = df["rsi_above_65"].rolling(6, min_periods = 1).sum()

    df["macd_signal"] = np.where(
        df["MACD"] > df["MACD_signal"], 1,
        np.where(df["MACD"] < df["MACD_signal"], -1, 0)
    )

    # El umbral 4 originalmente era 6 (AJUSTE para que salgan señales)
    df["sign_macd_rsi_robust"] = np.where((df["count_men"] >= 4) & (df["macd_signal"] == 1), 1,
                                          np.where((df["count_may"] >= 4) & (df["macd_signal"] == -1), -1, 0))
    del df["rsi_below_35"], df["rsi_above_65"], df["count_men"], df["count_may"], df["macd_signal"]


    #-----------1.2 MACD + RSI no exigente
    df["str_macd_rsi_simple"] = np.where((df["MACD"] > df["MACD_signal"]) & (df["RSI"] < 30), 1,
                                    np.where((df["MACD"] < df["MACD_signal"]) & (df["RSI"] > 70), -1, 0))

    #-----------1.1.2 MACD + RSI no exigente + confirmacion breakout
    df["str_macd_rsi_simple_break"] = np.where((df["str_macd_rsi_simple"] == 1) & (df["Close"] > df["High"].shift(1)), 1,
                                          np.where((df["str_macd_rsi_simple"] == -1) & (df["Close"] < df["Low"].shift(1)), -1, 0))

    # ---------- 2.2 MACD + RSI exigente + filtro de posición en BB
    df["bb_pos"] = bb_pos
    df["str_macd_rsi_simple_bb_pos"] = np.where((df["str_macd_rsi_simple"] == 1) & (df["bb_pos"] < 0.2), 1,
                                np.where((df["str_macd_rsi_simple"] == -1) & (df["bb_pos"] > 0.8), -1, 0))

    #---------- 3.1 RSI + Bollinger
    df["str_macd_rsi_bb"] = np.where((df["RSI"] < 30) & (df["Close"] < df["BB_lower"]), 1, 
                                   np.where((df["RSI"] > 70) & (df["Close"] > df["BB_upper"]), -1, 0))
    
    #---------- 3.2 RSI + Bollinger + Tendencia EMA
    df["str_macd_rsi_bb_ema"] = np.where((df["RSI"] < 30) & (df["Close"] < df["BB_lower"]) & (df["Close"] > df["EMA200"]), 1, 
                                   np.where((df["RSI"] > 70) & (df["Close"] > df["BB_upper"]) & (df["Close"] < df["EMA200"]), -1, 0))
    


    #---------------------------------- VERSIONES ADX
    #---------- 1 ADX + DI
    df["ind_adx"] = np.where((df["ADX"] > 25) & (df["ADX+DI"] > df["ADX-DI"]), 1, 
                     np.where((df["ADX"] > 25) & (df["ADX-DI"] > df["ADX+DI"]), -1, 0))
    
    #---------- 1.1 ADX + DI + Pendiente tendencia ADX
    # ADX > 25 puede venir de un movimiento viejo o ya agotado.
    # ADX creciente (ADX[t] > ADX[t-1])
    df["str_adx_crec"] = np.where((df["ind_adx"] == 1) & (df["ADX"] > df["ADX"].shift(1)), 1,
                          np.where((df["ind_adx"] == -1) & (df["ADX"] > df["ADX"].shift(1)), -1, 0))


    #---------- 1.1.2 ADX + DI + Pendiente tendencia ADX
    df["str_adx_crec_rsi"] = np.where((df["ind_adx"] == 1) & (df["ADX"] > df["ADX"].shift(1)) & (df["RSI"] < 45), 1,
                              np.where((df["ind_adx"] == -1) & (df["ADX"] > df["ADX"].shift(1)) & (df["RSI"] > 55), -1, 0))

    #---------- 1.1.3 ADX + DI + Pendiente tendencia ADX
    df["str_adx_rsi"] = np.where((df["ind_adx"] == 1) & (df["RSI"] < 40), 1,
                         np.where((df["ind_adx"] == -1) & (df["RSI"] > 60), -1, 0))

    df["str_adx_rsi_robust"] = np.where((df["ind_adx"] == 1) & (df["RSI"] < 30), 1,
                                np.where((df["ind_adx"] == -1) & (df["RSI"] > 70), -1, 0))
    
    df["str_adx_rsi_xtrem"] = np.where((df["ind_adx"] == 1) & (df["RSI"] < 20), 1,
                               np.where((df["ind_adx"] == -1) & (df["RSI"] > 80), -1, 0))



    # ---------- 2. VPVMA (volumen ponderado dinámico)
    """
    Qué mide: Precio ponderado por volumen vs su media
    Es parecido a VWAP, pero más “reactivo”
    Cruce = cambio en el equilibrio entre compradores/vendedores reales
    Detecta: “quién está ganando con dinero real”
    Riesgo: Sensible a spikes de volumen
    """

    signbuy_base = (
        (df["VPVMA"] > (1 + BANDWITH) * df["VPVMAS"]) &
        (df["VPVMA"].shift(1) <= df["VPVMAS"].shift(1))
    )

    signsell_base = (
        (df["VPVMA"] < (1 - BANDWITH * 2) * df["VPVMAS"]) &
        (df["VPVMA"].shift(1) >= df["VPVMAS"].shift(1))
    )

    window = 1
    recent_buy = signbuy_base.rolling(window).max().fillna(0).astype(bool)
    recent_sell = signsell_base.rolling(window).max().fillna(0).astype(bool)

    df["str_vpvma"] = np.where(recent_buy, 1, np.where(recent_sell, -1, 0))


    # ----------- 3. Sinergia
    """
    “solo opero cuando TODO está alineado”
    Riesgo
    Puede filtrar demasiado (pocas señales)
    """
    long_cond = (
        (df["Close"] > df["EMA20"]) &
        (df["RSI"] > 55) &
        (df["MACD"] > df["MACD_signal"]) &
        (df["Close"] > df["VWAP"]) &
        (df["spread"] < 0.01) &
        (df["volume_increasing"]) &
        (df["order_flow"] == "buy")
    )

    short_cond = (
        (df["Close"] < df["EMA20"]) &
        (df["RSI"] < 45) &
        (df["MACD"] < df["MACD_signal"]) &
        (df["Close"] < df["VWAP"]) &
        (df["spread"] < 0.01) &
        (df["volume_increasing"]) &
        (df["order_flow"] == "sell")
    )

    df["str_sinergia"] = 0
    df.loc[long_cond, "str_sinergia"] = 1
    df.loc[short_cond, "str_sinergia"] = -1
    del df["order_flow"], df["volume_increasing"], df["spread"]


    # ----------- 4. RSI mean reversion + Bollinger
    """
    Qué mide
    Desviación XTREM del precio respecto a la media
    Se predice que el precio hace una reversion a la media
    Interpretación: “el mercado se estiró demasiado, va a regresar”
    Riesgo: En tendencias fuertes, el precio sigue “rompiendo la banda”
    """

    df["str_rsi_bb_rever"] = np.where((df['RSI'] < 30) & (df['Close'] <= df['Low']), 1, 
                             np.where((df['RSI'] > 70) & (df['Close'] >= df['High']), -1, 0))


    # ----------- 5. Trend following: MA + MACD. 
    """
    Qué mide: Dirección + confirmación de momentum
    Interpretación: “me subo a la ola, no intento predecirla”
    Riesgo: Entradas tardías
    """
    
    df['trend'] = np.where(df['Close'] > df['EMA200'], 1, -1)

    df['long_signal'] = (
        (df['EMA50'] > df['EMA200']) &
        (df['MACD'] > df['MACD_signal']) &
        (df['trend'] == 1)
    )

    df['short_signal'] = (
        (df['EMA50'] < df['EMA200']) &
        (df['MACD'] < df['MACD_signal']) &
        (df['trend'] == -1)
    )

    # Evitar lookahead bias
    df['long_signal'] = df['long_signal'].shift(1)
    df['short_signal'] = df['short_signal'].shift(1)

    df["str_ma_macd"] = 0
    df.loc[df['long_signal'] == True, "str_ma_macd"] = 1
    df.loc[df['short_signal'] == True, "str_ma_macd"] = -1
    del df["long_signal"], df["short_signal"]



    # ----------- 6. Squeeze + Keltner + Bollinger
    """
    Qué mide: Contracción de volatilidad
    Interpretación: “el mercado está comprimido → se viene explosión”
    Riesgo: No siempre predice dirección
    """

    length = 20
    mult_bb = 2.0
    mult_kc = 1.5

    # Bollinger Bands
    df['ma_20'] = df['Close'].rolling(window=length).mean()
    df['std_20'] = df['Close'].rolling(window=length).std()
    df['bb_upper'] = df['ma_20'] + (mult_bb * df['std_20'])
    df['bb_lower'] = df['ma_20'] - (mult_bb * df['std_20'])

    # Keltner Channels (Usando ATR para la banda)
    df['keltner_channels'] = np.maximum(df['High'] - df['Low'], 
                            np.maximum(abs(df['High'] - df['Close'].shift(1)), 
                                        abs(df['Low'] - df['Close'].shift(1))))
    df['atr_20'] = df['keltner_channels'].rolling(window=length).mean()
    df['kc_upper'] = df['ma_20'] + (mult_kc * df['atr_20'])
    df['kc_lower'] = df['ma_20'] - (mult_kc * df['atr_20'])

    # Lógica del Squeeze
    df['sign_squeeze'] = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])

    # Momentum (Filtro simple para dirección del squeeze)
    df['momentum'] = df['Close'] - df['Close'].shift(length)

    # SE CAMBIO MOMENTUM POR ADX
    df["str_squ_kel_bb"] = np.where((df['sign_squeeze'] == True) & (df['ind_adx'] == 1), 1,  
                           np.where((df['sign_squeeze'] == True) & (df['ind_adx'] == -1), -1, 0))
    
    del df["momentum"], df["bb_upper"], df["bb_lower"], df["kc_upper"], df["kc_lower"], df["keltner_channels"]


    # ----------- 1. Golden Cross y Death Cross 
    """
    Qué mide: Cambio estructural de tendencia
    Interpretación: Es una señal “lenta pero fuerte”
    Riesgo: Muy retrasada
    """
    golden_base = (
        (df["EMA20"] > df["EMA50"]) &
        (df["EMA20"].shift(1) <= df["EMA50"].shift(1))
    )

    death_base = (
        (df["EMA20"] < df["EMA50"]) &
        (df["EMA20"].shift(1) >= df["EMA50"].shift(1))
    )

    window = 1
    recent_golden = golden_base.rolling(window).max().fillna(0).astype(bool)
    recent_death = death_base.rolling(window).max().fillna(0).astype(bool)

    df["sign_gold"] = np.where(recent_golden, 1, np.where(recent_death, -1, 0))


    # ----------- 3. 3-EMA alignment
    """
    Qué mide: Jerarquía de tendencias
    Interpretación: mercado ordenado vs caótico
    """
    df['ind_3ema'] = np.where((df['EMA20'] > df['EMA50']) & (df['EMA50'] > df['EMA200']), 1,  np.where((df['EMA20'] < df['EMA50']) & (df['EMA50'] < df["EMA200"]), -1, 0))


    # ----------- 4. Sobrecompra y sobreventa RSI
    df["sign_rsi30"] = np.where((df["RSI"] < 30), 1, np.where((df["RSI"] > 70), -1, 0))
    df["sign_rsi20"] = np.where((df["RSI"] < 20), 1, np.where((df["RSI"] > 80), -1, 0))
    df["sign_rsi15"] = np.where((df["RSI"] < 15), 1, np.where((df["RSI"] > 85), -1, 0))


    # ----------- 5. ADX + DI - Direccion y fuerza tendencia
    """
    Qué mide: Fuerza + dirección
    Interpretación: diferencia entre tendencia real vs ruido
    """
    def classify(row):
        adx_val   = row["ADX"]
        p_di      = row["ADX+DI"]
        m_di      = row["ADX-DI"]
        precio    = row["Close"]
        ema200    = row["EMA200"]

        if pd.isna(adx_val) or pd.isna(p_di) or pd.isna(m_di):
            return 0

        #  Mercado lateral: ADX muy bajo, sin dirección clara
        if adx_val < 15:
            return 0

        # Determinar dirección
        alcista = (p_di > m_di) and (precio > ema200)
        bajista = (m_di > p_di) and (precio < ema200)

        # Caso borde: DI's apuntan contra EMA200 (conflicto de señales)
        if not alcista and not bajista:
            if p_di > m_di:
                return 1   # alcista pero bajo EMA200
            elif m_di > p_di:
                return -1   # bajista pero sobre EMA200
            else:
                return 0

        # Clasificar por fuerza del ADX
        if alcista:
            if adx_val >= 70:
                return 5
            elif adx_val >= 60:
                return 4
            elif adx_val >= 40:
                return 3
            elif adx_val >= 25:
                return 2
            else:             
                return 1
        if bajista:
            if adx_val >= 70:
                return -5
            elif adx_val >= 60:
                return -4
            elif adx_val >= 40:
                return -3
            elif adx_val >= 25:
                return -2
            else:
                return -1

        return 0

    df["trend"] = df.apply(classify, axis=1)

    

    # ----------- 8. Pullback + EMA
    """
    Qué mide: Retroceso dentro de tendencia
    Interpretación: no persigo el precio, espero descuento
    Es lo que hacen traders institucionales: Compran en retrocesos. No en rompimientos emocionales

    Mejora el ratio riesgo/beneficio brutalmente. Reduce drawdowns
    """
    df["str_pullb_ema"] = np.where((df["Close"] > df["EMA50"]) & (df["Low"] <= df["EMA20"]) & (df["Close"] > df["EMA20"]), 1,
                          np.where((df["Close"] < df["EMA50"]) & (df["High"] >= df["EMA20"]) & (df["Close"] < df["EMA20"]), -1, 0))
    

    # ----------- 9. Divergencia RSI - agotamiento 
    """
    Qué mide: Desacople entre precio y momentum
    Interpretación: el precio sigue subiendo… pero con menos fuerza interna. Es como un coche subiendo una montaña 
    con el motor fallando.

    Muy potente en techos/suelos
    Mejor en temporalidades altas

    Precio hace lower low, RSI no → BUY
    Precio hace higher high, RSI no → SELL
    """
    df["price_diff"] = df["Close"].diff()
    df["rsi_diff"] = df["RSI"].diff()

    df["str_rsi_div"] =np.where((df["price_diff"] < 0) & (df["rsi_diff"] > 0), 1,
                       np.where((df["price_diff"] > 0) & (df["rsi_diff"] < 0), -1, 0))
    del df["price_diff"], df["rsi_diff"]


    # ----------- 10. VWAP Deviation
    """
    Muy lejos del VWAP → reversión probable
    Qué mide: Distancia respecto al precio promedio ponderado
    Interpretación: qué tan “caro” o “barato” está el precio en el día

    Institucionales usan VWAP como referencia. Grandes desviaciones → reversión probable
    """
    df["vwap_dev"] = (df["Close"] - df["VWAP"]) / df["VWAP"]

    df["ind_vwap_devi"] = np.where(df["vwap_dev"] < -0.02, 1, np.where(df["vwap_dev"] > 0.02, -1, 0))
    del df["vwap_dev"]


    # 11. ----------- Support / Resistance dinámico 
    """
    Qué mide: Zonas donde el precio ha reaccionado antes
    Interpretación: memoria del mercado. Los traders recuerdan esos niveles… y actúan ahí

    Más fuerte si coincide con volumen
    Más fuerte si coincide con estructura
    """
    n = 50
    lookback_break = 5  # máximo de velas desde el rompimiento

    df["resistance"] = df["High"].rolling(n).max()
    df["support"] = df["Low"].rolling(n).min()

    df["break_up"] = (df["Close"] > df["resistance"].shift(1))
    df["break_down"] = (df["Close"] < df["support"].shift(1))

    df["bars_since_break_up"] = (~df["break_up"]).cumsum() - (~df["break_up"]).cumsum().where(df["break_up"]).ffill()
    df["bars_since_break_down"] = (~df["break_down"]).cumsum() - (~df["break_down"]).cumsum().where(df["break_down"]).ffill()

    df["str_supo_resi"] = np.where(df["bars_since_break_up"] <= lookback_break, -1, 
                             np.where(df["bars_since_break_down"] <= lookback_break, 1, 0))
    del df["resistance"], df["support"], df["break_up"], df["break_down"], df["bars_since_break_up"], df["bars_since_break_down"]



    # ----------- 14. Momentum puro (Rate of Change)
    """
    Mide velocidad del precio (aceleración)
    Qué mide: velocidad del precio
    Interpretación: qué tan rápido se mueve el mercado

    Momentum precede a tendencia
    """
    n = 10
    df["ROC"] = df["Close"].pct_change(n)

    df["ind_momentum"] = np.where(df["ROC"] > 0.02, 1, np.where(df["ROC"] < -0.02, -1, 0))


    # # ----------- INDICATOR: 15. Volumen anómalo (Volume Spike)
    """
    Detecta actividad institucional o eventos
    Qué mide: volumen anormal
    Interpretación: actividad institucional o evento importante

    Confirmación clave de movimientos
    """
    df["vol_mean"] = df["Volume"].rolling(20).mean()

    df["ind_volume_anormal"] = np.where(df["Volume"] > 2 * df["vol_mean"], 1, 0)
    del df["vol_mean"]


    # ----------- INDICATOR: 16. Range Compression (antes de breakout)
    """
    Detecta consolidación XTREM
    Qué mide: reducción XTREM del rango
    Interpretación: mercado “apretado” antes de moverse

    Precede breakouts
    """
    df["range"] = df["High"] - df["Low"]
    df["range_mean"] = df["range"].rolling(20).mean()

    df["ind_compress_xtrem"] = np.where(df["range"] < 0.5 * df["range_mean"], 1, 0)
    del df["range"], df["range_mean"]


    # ----------- 17. EMA Distance (sobre-extensión)
    """
    Detecta cuando el precio está demasiado lejos de la media
    Qué mide: sobre-extensión del precio
    Interpretación: el precio se alejó demasiado de su equilibrio

    Evita entrar tarde
    Ideal para contrarian trades
    """
    df["ema_dist"] = (df["Close"] - df["EMA50"]) / df["EMA50"]

    df["sign_sobreext"] = np.where(df["ema_dist"] > 0.03, -1,
                          np.where(df["ema_dist"] < -0.03, 1, 0))


    # ----------- 19. Stochastic K/D crossover en zonas XTREMs
    """
    Tu sistema tiene Stochastic calculado pero comentado. Esta señal aprovecha el cruce K/D en zonas XTREMs.
    """
    df["sign_stochas"] = np.where((df["STOCH_K"] < 25) & (df["STOCH_D"] < 25) & (df["STOCH_K"].shift(1) < df["STOCH_D"].shift(1)) & (df["STOCH_K"] > df["STOCH_D"]), 1,
                         np.where((df["STOCH_K"] > 75) & (df["STOCH_D"] > 75) & (df["STOCH_K"].shift(1) > df["STOCH_D"].shift(1)) & (df["STOCH_K"] < df["STOCH_D"]), -1, 0))


    # ----------- 20. Chaikin Money Flow (CMF)
    # Mide la presión compradora/vendedora ponderada por volumen durante N períodos. Más sensible que el OBV a cambios intradía.
    CMF_PERIOD = 20
    CMF_UMBRAL = 0.05

    df["CLV"] = ((df["Close"] - df["Low"]) - (df["High"] - df["Close"])) / (df["High"] - df["Low"] + 1e-9)
    df["CMF"] = (df["CLV"] * df["Volume"]).rolling(CMF_PERIOD).sum() / df["Volume"].rolling(CMF_PERIOD).sum()

    df["sign_cmf"] = np.where((df["CMF"] > CMF_UMBRAL) & (df["CMF"] > df["CMF"].shift(1)), 1, 
                     np.where((df["CMF"] < -CMF_UMBRAL) & (df["CMF"] < df["CMF"].shift(1)), -1, 0))
    del df["CLV"], df["CMF"]



    # ----------- 21. Parabolic SAR
    # Excelente para gestión de stops dinámicos y detección de reversiones. Complementa al Supertrend — juntos forman una doble confirmación ATR-based.
    def calc_psar(df, iaf=0.02, maxaf=0.2):

        df = df.copy()

        length = len(df)
        high = df["High"].values
        low = df["Low"].values

        psar = np.zeros(length)

        bull = True
        af = iaf
        hp = high[0]
        lp = low[0]

        # Inicialización
        psar[0] = low[0]
        psar[1] = low[0]

        for i in range(2, length):

            if bull:
                psar[i] = psar[i-1] + af * (hp - psar[i-1])
                psar[i] = min(psar[i], low[i-1], low[i-2])

                if low[i] < psar[i]:
                    bull = False
                    psar[i] = hp
                    lp = low[i]
                    af = iaf
                else:
                    if high[i] > hp:
                        hp = high[i]
                        af = min(af + iaf, maxaf)

            else:
                psar[i] = psar[i-1] + af * (lp - psar[i-1])
                psar[i] = max(psar[i], high[i-1], high[i-2])

                if high[i] > psar[i]:
                    bull = True
                    psar[i] = lp
                    hp = high[i]
                    af = iaf
                else:
                    if low[i] < lp:
                        lp = low[i]
                        af = min(af + iaf, maxaf)

        df["PSAR"] = psar

        # Señal
        df["ind_psar"] = np.where(df["Close"] > df["PSAR"], 1, np.where(df["Close"] < df["PSAR"], -1, 0))

        return df["ind_psar"], df["PSAR"]
    
    df["ind_psar"], df["PSAR"] = calc_psar(df)


    # # ----------- 22. Elder Ray Index
    # Mide la diferencia entre el precio y una EMA. El Bull Power es High - EMA13, Bear Power es Low - EMA13. 
    # Detecta si los compradores/vendedores están ganando fuerza dentro de la tendencia.

    EMA_ELDER = 13
    df["EMA13"]      = df["Close"].ewm(span=EMA_ELDER, adjust=False).mean()
    df["bull_power"] = df["High"] - df["EMA13"]
    df["bear_power"] = df["Low"]  - df["EMA13"]

    # Señal: tendencia alcista con bear_power cruzando hacia positivo (compresión terminando)
    df["sign_eri"] = np.where((df["bull_power"].shift(1) > 0) & (df["bear_power"].shift(1) < 0) & (df["bear_power"].shift(2) < df["bear_power"].shift(1)), 1,
                     np.where((df["bull_power"].shift(1) < 0) & (df["bear_power"].shift(1) > 0) & (df["bull_power"].shift(2) < df["bull_power"].shift(1)), 1, 0))

    del df["EMA13"], df["bull_power"], df["bear_power"] 


    # ----------- 23. Williams %R
    # Similar al Stochastic pero más reactivo. Útil para detectar zonas de agotamiento rápido, especialmente en intradía.
    WILLR_PERIOD = 14
    highest_high = df["High"].rolling(WILLR_PERIOD).max()
    lowest_low   = df["Low"].rolling(WILLR_PERIOD).min()
    df["WillR"]  = -100 * (highest_high - df["Close"]) / (highest_high - lowest_low + 1e-9)

    # Saliendo de zonas XTREMs (más preciso que entrar en ellas)
    df["sign_willr"] = np.where((df["WillR"].shift(-2) < -80) & (df["WillR"].shift(-1) > -80), 1,   # saliendo de sobreventa
                        np.where((df["WillR"].shift(-2) > -20) & (df["WillR"].shift(-1) < -20), -1, 0)) # saliendo de sobrecompra


    # ----------- 25. Donchian Channel Breakout
    """
    Captura rupturas de rango después de consolidación. Complementa perfectamente al Squeeze porque el Squeeze dice cuándo viene el movimiento y Donchian dice si ya rompió.
    Detecta ruptura de máximos/mínimos recientes (tendencia fuerte)
    """
    n = 20
    df["donchian_high"] = df["High"].rolling(n).max()
    df["donchian_low"]  = df["Low"].rolling(n).min()

    df["sign_donchian"] = np.where(df["Close"] > df["donchian_high"].shift(1), 1,
                          np.where(df["Close"] < df["donchian_low"].shift(1), -1, 0))


    # ----------- 26. Liquidity Sweep (caza de stops)
    #  Muy usado por smart money.
    # Lógica
    # Rompe máximo previo pero cierra por debajo → trampa → SELL
    # Rompe mínimo previo pero cierra por encima → BUY

    df["prev_high"] = df["High"].shift(1)
    df["prev_low"] = df["Low"].shift(1)

    df["sign_liquid"] = np.where((df["High"] > df["prev_high"]) & (df["Close"] < df["prev_high"]), -1, 
                        np.where((df["Low"] < df["prev_low"]) & (df["Close"] > df["prev_low"]), 1, 0))

    del df["prev_high"], df["prev_low"]



    # ----------- 28. Divergencia MACD
    def compute_macd(df, fast=12, slow=26, signal=9):
        ema_fast = df["Close"].ewm(span=fast, adjust=False).mean()
        ema_slow = df["Close"].ewm(span=slow, adjust=False).mean()

        macd = ema_fast - ema_slow
        macd_signal = macd.ewm(span=signal, adjust=False).mean()
        hist = macd - macd_signal

        df = df.copy()
        df["MACD"] = macd
        df["MACD_signal"] = macd_signal
        df["MACD_hist"] = hist

        return df


    def find_pivots(series, left=3, right=3):
        pivots_high = []
        pivots_low = []

        for i in range(left, len(series) - right):
            window = series[i-left:i+right+1]

            if series.iloc[i] == window.max():
                pivots_high.append(i)

            if series.iloc[i] == window.min():
                pivots_low.append(i)

        return pivots_high, pivots_low


    def detect_macd_divergence(df, left=3, right=3):

        df = compute_macd(df)

        price = df["Close"]
        macd = df["MACD"]

        highs, lows = find_pivots(price, left, right)

        signals = pd.DataFrame(index=df.index)
        signals["bullish_div"] = False
        signals["bearish_div"] = False

        for i in range(1, len(lows)):
            prev = lows[i-1]
            curr = lows[i]
            confirm = curr + right          # vela donde el pivote ya es observable
            if confirm >= len(df):
                continue
            price_ll = price.iloc[curr] < price.iloc[prev]
            macd_hl = macd.iloc[curr] > macd.iloc[prev]
            if price_ll and macd_hl:
                signals.loc[df.index[confirm], "bullish_div"] = True

        for i in range(1, len(highs)):
            prev = highs[i-1]
            curr = highs[i]
            confirm = curr + right
            if confirm >= len(df):
                continue
            price_hh = price.iloc[curr] > price.iloc[prev]
            macd_lh = macd.iloc[curr] < macd.iloc[prev]
            if price_hh and macd_lh:
                signals.loc[df.index[confirm], "bearish_div"] = True

        return signals["bearish_div"], signals["bullish_div"]

    df["bearish_div"], df["bullish_div"] = detect_macd_divergence(df)

    df["tempsignal"] = np.where(df["bullish_div"], 1, np.where(df["bearish_div"], -1, 0))

    #----------- V1
    window = 1
    recent_signal = df["tempsignal"].replace("", np.nan).ffill(limit=window)
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div1"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")

    #----------- V2
    window = 3
    recent_signal = (df["tempsignal"].replace("", np.nan).ffill(limit=window))
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div3"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")

    #----------- V3
    window = 5
    recent_signal = (df["tempsignal"].replace("", np.nan).ffill(limit=window))
    df["str_macd_div"] = recent_signal.fillna("")
    df["str_macd_div5"] = df["str_macd_div"].where(df["str_macd_div"] != df["str_macd_div"].shift(1), "")


    del df["tempsignal"], df["bullish_div"], df["bearish_div"], df["str_macd_div"]



    #---------------------------------- OTHER STRATEGIES --------------------------------#


    # Random signals
    import random
    df["sign_random"] = np.random.choice([-1,0,1], len(df), p=[0.05, 0.9, 0.05])

    df["sign_random2"] = np.random.choice([-1,0,1], len(df), p=[0.025, 0.95, 0.025])

    df["sign_random3"] = np.random.choice([-1,0,1], len(df), p=[0.1, 0.8, 0.1])

    df["symbol"] = symbol

    df = df.iloc[40:] # Algunos indicadores no tienen informacion en primeros registros

    return pd.DataFrame(), df 


from joblib import Parallel, delayed

def main(SYMBOLS, INTERVAL, LOOKBACK_DAYS):
    print(f"Se analizan {len(SYMBOLS)} símbolos")
    print(f"Intervalo: {INTERVAL}  |  Lookback: {LOOKBACK_DAYS} días")

    output = []
    chunk_size = 90
    for i in range(0, len(SYMBOLS), chunk_size):
        chunk = SYMBOLS[i:i + chunk_size]
        chunk_results = Parallel(n_jobs=-1)(
            delayed(process_signals)(sym, INTERVAL, LOOKBACK_DAYS)
            for sym in chunk)
        output.extend(chunk_results)

    valid_rows = [row for row in output if row is not None and len(row) == 2]
    df = pd.concat([row[1] for row in valid_rows if row[1] is not None], ignore_index=True)

    return df

In [ ]:
symb = "ETH-USD"
save = False
_, df = process_signals(symb, "1h", 400) # Max 730
# FECHA_INICIO = '2025-04-01'  
# FECHA_FIN    = '2026-04-25'      
# df   = df[(df['date'] >= FECHA_INICIO) & (df['date'] <= FECHA_FIN)]

cols_str = [col for col in df.columns if col.startswith("str_")]
cols_sign = [col for col in df.columns if col.startswith("sign_")]
cols_strsig = cols_str + cols_sign
cols_exclude = [] 
cols_others = [] + cols_exclude
cols_strsig = [x for x in cols_strsig if x not in cols_others]

cols_base = [x for x in df if x not in cols_strsig] 
order = cols_base + sorted(cols_strsig)
df = df[order]

print("Total strategies-signals:", len(cols_strsig))
print("Strategies:", len(cols_str))
print("Signals:   ", len(cols_sign))
print("Others:    ", len(cols_base))

for signal in cols_strsig:
    try:
        mapping = {-1: -1, "": 0, 1: 1}
        df[signal] = df[signal].replace(mapping).fillna(0).astype(int)
    except:
        print("ERROR:", signal)
        
df2 = df.copy()
df

In [ ]:
res = []
for j in cols_strsig:
    buys = df[df[j].isin([1])].shape[0]
    sells = df[df[j] == -1].shape[0] 
    n = buys + sells
    res.append(pd.DataFrame({"signal":[j], "n":[n], "%":[(n/len(df))*100], "buys":[buys], "buys%":[(buys/len(df))*100],
                              "sells":[sells],"sells%":[(sells/len(df))*100]}))
print(len(df))
res = pd.concat(res).sort_values("%", ascending = False)
res.head(20)

In [ ]:
revisar las señales que tienen 0% en sell o buy

In [ ]:
qwe

## Backtester

In [ ]:
# Parameters
risk_per_trade        = 0.01
fee_rate              = 0.0007
slippage_pct          = 0.0005
atr_period            = 14
# min_holding_period    = 0
leverage              = 100     # 10 = 1:10
# max_loss_streak_limit = 5
# cooldown_after_losses = 10
# ST y TP
atr_sl                = 2        # Value for SL. Scalping: 0.8-1.5 | Intradia: 1.5-2.5 | Swing: 2-3.5 | Largo: 3-5
multiplier_tp         = 1.5        # Value for TP -> atr_tp = atr_sl x multiplier_tp. 
use_r_tp              = False    # Si se activa, no se usara ATR si no R para fijar sl, tp. R: distancia a sl, para calcular tp. 
factor_r_tp           = 2        # Factor para fijar TP = entry price + factor_r_tp * R. R=price - sl. Conservador: 1-1.5 | Balanceado: 2 | Tendencial | 3-5
# Trailing
use_trailing_stop     = False
use_breakeven         = use_trailing_stop  # Siempre usar
trailing_atr_mult     = 1                  # default=2. Value por trailing stop
trailing_activation_r = 2.5                # default=2. Solo activar trailing después de 2R de ganancia, el break-even se activa antes, al llegar a 1R. Evitar que sea igual que breakeven_trigger_r
breakeven_trigger_r   = 3                  # default=1. break-even al llegar a 1R si valor = 0 -> stop loss se posicione en precio de entrada. Cero ganancias


# ── Gestión de riesgo de cartera ─────────────────────────────
# max_dd_daily_pct       = 0.05        # parar si el día pierde >5%
# cooldown_bars          = 3           # barras de espera tras un SL
# max_trades_per_day     = 6

In [ ]:
def def_engine(df, name_signal):
    resume = []
    state = {}
    
    df = df.copy()

    capital = 1000
    patrimony = capital 
    position_opened = False
    candle_trade = []
    state = []
    idx = 0
    bars_held = np.nan
    cooldown = 0
    n = len(df)

    df["signal"] = df[name_signal]

    for i in range(n):

        bar        = df.iloc[i]
        prev_bar   = df.iloc[i - 1]
        signal     = bar["signal"]
        high       = bar["High"]
        low        = bar["Low"]
        atr        = bar["ATR"]
        price_var  = "Close"
        price      = bar[price_var]

        # COOLDOWN CONTROL
        if cooldown > 0:
            cooldown -= 1
            signal = 0  # bloquear entradas


        #-------------- 1. ENTRY
        if not position_opened and signal != 0:

            if signal == 1: 
                entry_price = price * (1 + slippage_pct)
            else:
                entry_price = price * (1 - slippage_pct)

            sl_distance = atr_sl * atr

            if signal == 1:
                stop_loss = entry_price - sl_distance
            else:
                stop_loss = entry_price + sl_distance

            tp_distance = abs(entry_price - stop_loss) * multiplier_tp

            if signal == 1:
                take_profit = entry_price + tp_distance
            else:
                take_profit = entry_price - tp_distance

            amount = patrimony * risk_per_trade
            patrimony = patrimony - amount

            risk_unit = abs(entry_price - stop_loss)
            size = amount / risk_unit
            position_value = size * entry_price
            max_position_value = amount * leverage      
            if position_value > max_position_value:
                size = max_position_value / entry_price

            # Niveles auxiliares (se calculan UNA vez en ENTRY)
            r_unit        = abs(entry_price - stop_loss)
            be_price      = entry_price + breakeven_trigger_r * r_unit * signal

            bars_held = 0

            temp = {
                "id": [idx],
                "time": [bar.name],
                "signal": [signal],
                "signal_trade": [signal],
                "close": [entry_price],
                "high": [high],
                "low": [low],
                "entry_price": [entry_price],
                "sl": [stop_loss],
                "sl_inicial":     [stop_loss],   # guardamos el SL original para calcular R
                "tp": [take_profit],
                "size": size,
                "bars_held": [bars_held],
                "entry_amount":[amount],
                "be_price":       [be_price],
                "breakeven_hit":  [False],
                "partial_hit":    [False],
                "trailing_active":[False],
                "price_prev":[prev_bar[price_var]],
                "diff":[bar[price_var] - prev_bar[price_var]],
                "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
                "diff_open": [np.nan],
                "delta_open":[np.nan],  
                "patrimony": [patrimony],
                "state": ["ENTRY"],
                "RSI":[bar["RSI"]],
                "ADX":[bar["ADX"]],
                "ADX+DI":[bar["ADX+DI"]],
                "ADX-DI":[bar["ADX-DI"]],
                "ATR":[bar["ATR"]]
            }
            temp = pd.DataFrame(temp)
            state.append(temp)
            position_opened = True
            candle_trade = temp.copy()
            pass


        #-------------- 2. MANAGE TRADE
        if position_opened:
            _exit        = False
            exitintrail  = False
            reason       = np.nan

            _close = bar["Close"]
            _high = bar["High"]
            _low = bar["Low"]
            _sl = candle_trade["sl"][0]
            _tp = candle_trade["tp"][0]
            _entry_amount = candle_trade["entry_amount"][0]
            _entry_price = candle_trade["entry_price"][0]
            _size = candle_trade["size"][0]        
            signal_trade    = candle_trade["signal_trade"][0]
            _sl_inicial     = candle_trade["sl_inicial"][0]
            _be_price       = candle_trade["be_price"][0]
            _breakeven_hit  = candle_trade["breakeven_hit"][0]
            _trailing_active= candle_trade["trailing_active"][0]
            _close          = bar["Close"]


            # REVERSION OF SIGNAL CONDITION
            if signal != 0 and signal != signal_trade:
                _exit = True
                reason = "REV"


            temp = candle_trade.copy()

            temp["diff_open"] = (bar[price_var] - candle_trade[price_var.lower()])[0]
            temp["delta_open"] = ((bar[price_var] - candle_trade[price_var.lower()]) /
                                candle_trade[price_var.lower()])[0]
            
            # P&L actual de la posición (sobre valor real apalancado)
            actual_pos_value = _size * _entry_price          
            delta_open       = (_close - _entry_price) / _entry_price
            pnl              = actual_pos_value * delta_open * signal_trade

            # BREAK-EVEN
            if use_breakeven and not _breakeven_hit:
                be_hit = (signal_trade ==  1 and _close >= _be_price) or \
                         (signal_trade == -1 and _close <= _be_price)
                if be_hit:
                    _sl = _entry_price
                    candle_trade["sl"]           = _sl
                    candle_trade["breakeven_hit"]= True
                    _breakeven_hit               = True

            # TRAILING STOP (solo se activa después de trailing_activation_r)
            if use_trailing_stop:
                r_unit       = abs(_entry_price - _sl_inicial)
                ganancia_en_r = abs(_close - _entry_price) / r_unit if r_unit != 0 else 0

                if ganancia_en_r >= trailing_activation_r:
                    candle_trade["trailing_active"] = True
                    _trailing_active = True

                if _trailing_active:
                    trail_dist = trailing_atr_mult * atr

                    if signal_trade == 1:                     
                        new_trail = high - trail_dist
                        if new_trail > _sl:
                            _sl = new_trail
                            candle_trade["sl"] = _sl
                            exitintrail = True
                    else:
                        new_trail = low + trail_dist
                        if new_trail < _sl:
                            _sl = new_trail
                            candle_trade["sl"] = _sl
                            exitintrail = True


            if signal_trade == 1:
                sl_hit = low <= _sl
                tp_hit = high >= _tp
                if sl_hit:
                    _exit = True
                    reason = "SL"
                    reason = "TS" if reason == "SL" and exitintrail else reason
                elif tp_hit:
                    _exit = True
                    reason = "TP"


            if signal_trade == -1:
                sl_hit = high >= _sl
                tp_hit = low <= _tp
                if sl_hit:
                    _exit = True
                    reason = "SL"
                    reason = "TS" if reason == "SL" and exitintrail else reason
                elif tp_hit:
                    _exit = True
                    reason = "TP"


            # EXIT TRADE
            if _exit and bars_held >= 1:
                exit_price = _tp if reason == "TP" else _sl

                if reason == "REV":
                    exit_price = price  # cerrar al precio actual

                if signal_trade == 1:
                    _return = (exit_price - _entry_price) / _entry_price
                else:
                    _return = (_entry_price - exit_price) / _entry_price
                
                actual_position_value = _size * _entry_price          
                gross_profit = _return * actual_position_value          
                fees         = actual_position_value * fee_rate * 2       
                profit_f     = gross_profit - fees
                patrimony    += _entry_amount + profit_f


                temp = {
                    "id": [idx],
                    "time": [bar.name],
                    "close": [price],
                    "high": [high],
                    "low": [low],
                    "signal_trade":[0],
                    "price_prev":prev_bar[price_var],
                    "diff":bar[price_var] - prev_bar[price_var],
                    "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
                    "patrimony": [patrimony],
                    "state":"EXIT",
                    "reason":[reason],
                    "pnl":[_entry_amount + profit_f],
                    "entry_amount":[_entry_amount],
                    "profit_ex":[profit_f],
                    "return_ex":[_return],
                    "bars_held_ex":[bars_held],
                    "RSI":[bar["RSI"]],
                    "ADX":[bar["ADX"]],
                    "ADX+DI":[bar["ADX+DI"]],
                    "ADX-DI":[bar["ADX-DI"]],
                    "ATR":[bar["ATR"]],
                    "breakeven_hit":[_breakeven_hit],
                    "be_price":[_be_price],
                    "sl":[_sl],
                    "tp":[_tp],
                    
                }
                temp = pd.DataFrame(temp)
                state.append(temp)

                _exit = False
                position_opened = False
                candle_trade = []

                pass

            else:
                temp = candle_trade.copy()
                temp["id"] = idx
                temp["signal"] = np.nan
                temp["close"] = price
                temp["high"] = high
                temp["low"] = low
                temp["time"] = bar.name
                temp["bars_held"] = [bars_held]
                temp["price_prev"] = prev_bar[price_var]
                temp["diff"] = bar[price_var] - prev_bar[price_var]
                temp["delta"] = (bar[price_var] - prev_bar[price_var]) / prev_bar[price_var]
                temp["diff_open"] = (bar[price_var] - candle_trade[price_var.lower()])[0]
                temp["delta_open"] = ((bar[price_var] - candle_trade[price_var.lower()]) / candle_trade[price_var.lower()])[0]
                temp["patrimony"] = patrimony
                temp["state"] = "HOLD"
                temp["pnl"] = pnl
                temp["RSI"] = bar["RSI"]
                temp["ADX"] = bar["ADX"]
                temp["ADX+DI"] = bar["ADX+DI"]
                temp["ADX-DI"] = bar["ADX-DI"]
                temp["ATR"] = bar["ATR"]
                temp["breakeven_hit"] = _breakeven_hit
                temp["be_price"] = _be_price

                state.append(temp)

                pass


        if not position_opened and signal == 0:
            temp = {
                "id": [idx],
                "time": [bar.name],
                "signal_trade": [signal],
                "close": [price],
                "high": [high],
                "low": [low],
                "price_prev":prev_bar[price_var],
                "diff":bar[price_var] - prev_bar[price_var],
                "delta":[(bar[price_var] - prev_bar[price_var])/prev_bar[price_var]],
                "patrimony": [patrimony],
                "state":"NO SIGNAL",
                "RSI":[bar["RSI"]],
                "ADX":[bar["ADX"]],
                "ADX+DI":[bar["ADX+DI"]],
                "ADX-DI":[bar["ADX-DI"]],
                "ATR":[bar["ATR"]]
            }
            temp = pd.DataFrame(temp)
            state.append(temp)

            pass

        bars_held += 1
        idx += 1

    state = pd.concat(state)
    state.drop_duplicates("time", inplace = True) # Ajustar

    # Devolviendo capital invertido a la ultima vela si finalizando se tiene una operacion abierta y esta no cerró
    last = state.iloc[-1]
    indeter_signal = False
    if last["state"] == "HOLD":
        indeter_signal = True
        state["patrimony"] = np.where(state["id"] == last["id"], state["patrimony"] + state["entry_amount"], state["patrimony"])

    if "profit_ex" not in state:
        state["profit_ex"] = np.nan
    if "return_ex" not in state:
        state["return_ex"] = np.nan

    state["profit"] = state["pnl"] - state["entry_amount"]
    state["return%"] = state["profit"] / state["entry_amount"]
    state["profit"] = np.where(state["state"] == "EXIT", state["profit_ex"], state["profit"])
    state["return%"] = np.where(state["state"] == "EXIT", state["return_ex"], state["return%"])
    state["peak"] = state["patrimony"].cummax()
    state["drawdown"] = (state["patrimony"] - state["peak"]) / state["peak"]
    exits = state[state["state"] == "EXIT"].copy()

    returns = state.loc[state["state"] == "EXIT", "return%"].dropna()
    returns_pct = exits["return%"]

    if len(returns) > 1 and returns.std() != 0:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(len(returns))
    else:
        sharpe = np.nan

    max_dd = state["drawdown"].min()

    capital_f = state["patrimony"].iloc[-1]
    pnl_f = capital_f - capital
    return_f = ((capital_f / capital) - 1)*100
    exits["return%"] = exits["return%"]*100
    n_trades = exits.shape[0] if not indeter_signal else exits.shape[0] - 1
    win_rate = (exits[(exits["profit"] > 0)].shape[0] / n_trades)*100 if n_trades != 0 else 0

    days = (state["time"].max() - state["time"].min()) / pd.Timedelta(days=1)
    downside = exits["return%"]
    downside = downside[downside < 0]
    sortino = (exits["return%"].mean() / downside.std()) * np.sqrt(24*365) 
    gains = exits[exits["profit"] > 0]
    loss = exits[exits["profit"] < 0]
    exits["time_t1"] = exits["time"].shift(1)
    exits["diff_time"] = (exits["time"] - exits["time_t1"]) / pd.Timedelta(days=1)
    cagr   = ((capital_f / capital) ** (365 / max(days, 1)) - 1) * 100
    calmar = cagr / abs(max_dd * 100) if max_dd != 0 else np.nan
    win_rate_dec = len(gains) / n_trades if n_trades > 0 else 0
    max_loss_streak, cur_streak = 0, 0
    for s in exits["profit"]:
        cur_streak      = cur_streak + 1 if s < 0 else 0
        max_loss_streak = max(max_loss_streak, cur_streak)
    max_gain_streak, cur_streak = 0, 0
    for s in exits["profit"]:
        cur_streak      = cur_streak + 1 if s > 0 else 0
        max_gain_streak = max(max_gain_streak, cur_streak)
    hold_bars  = state[state["state"] == "HOLD"].shape[0]
    exposure   = hold_bars / len(state) * 100
    var_95  = np.percentile(returns_pct, 5)  if len(returns_pct) > 0 else np.nan
    cvar_95 = returns_pct[returns_pct <= var_95].mean() if len(returns_pct) > 0 else np.nan
    recovery_factor = pnl_f / abs(max_dd * capital) if max_dd != 0 else np.nan
    n_sl = exits[exits["reason"] == "SL"].shape[0]
    n_tp = exits[exits["reason"] == "TP"].shape[0]
    n_ts = exits[exits["reason"] == "TS"].shape[0]

    print("─────────────── Symbol ──────────────")
    print(f"               {symb}")

    print("\n───────── Time and Candles ──────────")
    print("Candles:", f"{len(df)} | Days: {int(days)}") # {window}
    print("Trades:           ", n_trades, f"| Wins: {len(gains)} | Loss: {len(loss)}")
    print("Trades/day:       ", f"{(n_trades/days):.1f}")
    print("Time/trade:       ", f"Mean: {exits["bars_held_ex"].mean():.0f} | Min: {exits["bars_held_ex"].min():.0f} | Max: {exits["bars_held_ex"].max():.0f}")
    print("Time inter trades:", f"Mean: {exits["diff_time"].mean():.0f} | Min: {exits["diff_time"].min():.0f} | Max: {exits["diff_time"].max():.0f}")

    print("\n─────────── Final Results ───────────")
    print("Win rate:         ", f"{win_rate:.0f}%")
    print("Capital initial:  ", f"${capital}")
    print("Capital final:    ", f"${capital_f:.1f}")
    print("PnL:              ", f"${pnl_f:.1f}")
    print("Return:           ", f"{(return_f):.1f}%")
    print("Max drawdown:     ", f"{max_dd:.1%}\n")

    print("Sharpe:           ", f"{sharpe:.1f}")
    print("Sortino:          ", f"{sortino:.0f}")
    print("Calmar:           ", f"{calmar:.0f}")
    print("CAGR:             ", f"{cagr:.0f}\n")

    print("Total gains:      ", f"${gains["profit"].sum():.0f}")
    print("Total loss:       ", f"${loss["profit"].sum():.0f}")
    print(" ")
    print("Return/trade:     ", f"Mean: {exits["return%"].mean():.0f}% | Min: {exits["return%"].min():.0f}% | Max: {exits["return%"].max():.0f}%")
    print("Profit/trade:     ", f"Mean: ${exits["profit"].mean():.1f} | Min: ${exits["profit"].min():.1f} | Max: ${exits["profit"].max():.1f}")

    print("\n─────────── Metrics Risk ───────────") # Risk:Reward Ratio
    exits["return_on_margin"] = exits["profit"] / exits["entry_amount"] * 100 # Retorno promedio real sobre margen (no sobre precio)
    avg_win = exits[exits['profit']>0]['return%'].mean()
    avg_loss = exits[exits['profit']<0]['return%'].mean()
    avg_win_margin = exits[exits['profit']>0]['return_on_margin'].mean()
    avg_loss_margin = exits[exits['profit']<0]['return_on_margin'].mean()
    loss_rate = 1 - win_rate   
    expectancy = (win_rate_dec * avg_win) - (loss_rate * avg_loss)
    prom_fee = exits['entry_amount'].mean() * fee_rate * 2 * leverage
    print(f"Prom Position Value:${exits['entry_amount'].mean() * leverage:.0f} (with leverage)")
    print(f"Prom Position Value:${exits['entry_amount'].mean():.0f} (without leverage)")

    print(f"Expectancy:         {expectancy:.1f}")
    print(f"Avg win (price):    {avg_win:.1f}%")
    print(f"Avg loss (price):   {avg_loss:.1f}%\n")

    print(f"Ratio W/L:          {abs(avg_win / avg_loss):.1f}x | Gain prom = {abs(avg_win / avg_loss):.1f} x Loss") # en promedio cada ganancia es 2.6 veces más grande que cada pérdida.
    print(f"Recovery factor:    {recovery_factor:.2f}")
    print(f"VaR 95%:            {var_95:.1f}%")
    print(f"CVaR 95%:           {cvar_95:.1f}%\n")

    print(f"Max racha loss:     {max_loss_streak:.0f}")
    print(f"Max racha gain:     {max_gain_streak:.0f}\n")

    clos_tp = n_tp/n_trades*100 if n_trades != 0 else 0
    clos_sl = n_sl/n_trades*100 if n_trades != 0 else 0
    clos_ts = n_ts/n_trades*100 if n_trades != 0 else 0

    print(f"Closed x TP:        {n_tp} ({clos_tp:.0f}%)")
    print(f"Closed x SL:        {n_sl} ({clos_sl:.0f}%)")
    print(f"Closed x TS:        {n_ts} ({clos_ts:.0f}%)\n")

    print(f"Avg win (margin):   {avg_win_margin:.0f}%")
    print(f"Avg loss (margin):  {avg_loss_margin:.0f}%\n")

    print(f"Fees estim/trade:   ${prom_fee:.1f}")
    print(f"Total fees estim:   ${prom_fee * n_trades:.0f}")
    print(f"Fees as %avg_win:   {prom_fee / exits[exits['profit']>0]['profit'].mean() * 100:.0f}%\n")

    resume = pd.DataFrame({"symbol":[symb], "signal":[name_signal], "capital":[capital_f], "win_rate":[win_rate], "pnl":[pnl_f], "return":[return_f], 
                        "trades":[n_trades], "tradesxday":[n_trades/days], "drawdown":[max_dd], "sharpe":[max_dd], "sharpe":[sharpe], "ratio_wl":[abs(avg_win / avg_loss)],
                        "total_gains":[gains["profit"].sum()], "total_loss":[loss["profit"].sum()], "mean_timextrade":[exits["bars_held_ex"].mean()],
                        "mean_profit":[exits["profit"].mean()], "min_profit":[exits["profit"].min()],"max_profit":[exits["profit"].mean()],
                        "mean_position":[exits['entry_amount'].mean() * leverage], "expectancy":[expectancy], "recov_factor":[recovery_factor], "max_rach_loss":[max_loss_streak],
                        "max_rach_gain":[max_gain_streak], "close_tp":[clos_tp], "close_sl":[clos_tp],"close_ts":[clos_tp],
                        "mean_fees":[prom_fee], "total_fees":[prom_fee * n_trades], "param.atr_sl":[atr_sl], "param.multiplier_tp":[multiplier_tp], 
                        "param.trailing_atr_mult":[trailing_atr_mult], "param.use_trailing_stop":[use_trailing_stop], "param.use_breakeven":[use_breakeven],
                        "param.trailing_activation_r":[trailing_activation_r], "param.breakeven_trigger_r":[breakeven_trigger_r],
                        "param.use_r_tp":[use_r_tp], "param.factor_r_tp":[factor_r_tp],
                        })

    state.drop(columns = ["diff_open", "price_prev", "diff", "delta", "profit_ex", "return_ex", "peak", "bars_held_ex", "delta_open",# "be_price", "breakeven_hit",
                          "trailing_active", "sl_inicial"], inplace =True)
    return resume, state
    print("="*120)
    print("="*120, "\n")


## 1. Tester de varias señales en un simbolo

In [ ]:
FECHA_INICIO = '2026-01-01'  
FECHA_FIN    = '2026-12-01'  

df = df2.copy()    
df   = df[(df['date'] >= FECHA_INICIO) & (df['date'] <= FECHA_FIN)]
print(df.shape)
print(df["date"].min() , "|" ,df["date"].max())
df.tail(10)

In [ ]:
print(len(cols_strsig))
df[cols_strsig].tail(5)

In [ ]:
def run_signal(name_signal):
    try:
        res, state = def_engine(df, name_signal)
    except:
        res = pd.DataFrame()
        state = pd.DataFrame()
        
    return name_signal, res, state

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_signal)(name_signal) for name_signal in cols_strsig
)

resume = []
state = {}

for name_signal, res, state in results:
    resume.append(res)
    state[name_signal] = state
    
resume = pd.concat(resume).sort_values("capital", ascending = False)
resume

## 2. Test de todas las señales en todos los simbolos

* Paso 1: Correr en evaluacion para seleccionar simbolos y señales
* Paso 2: Hacer backtesting de lo seleccionado

In [ ]:
porc_eval = 0.7 
DAYS = 220  # Max 730

# Revisando fechas para fijar eval y backt
df = yf.download("BTC-USD", period=f"{DAYS}d", interval="1h", progress=False, auto_adjust=True)
df.columns = df.columns.get_level_values(-1)
df = df.reset_index()

df["mes"] = df["Datetime"].dt.strftime("%Y-%m-%d")
df = df.drop_duplicates("mes")
c = int(len(df)*porc_eval)
# print(c)

dates_eval = [df.iloc[:c]["mes"].tolist()[0], df.iloc[:c]["mes"].tolist()[-1]]
dates_predict = [df.iloc[c:]["mes"].tolist()[0], df.iloc[c:]["mes"].tolist()[-1]]

dates = dates_eval  # Cambiar a necesidad

print("Eval:", dates_eval)
print("Pred:", dates_predict)

In [ ]:
# import shutil, os

# shutil.rmtree(f"{path_data}resume/")
# os.makedirs(f"{path_data}resume/")

# shutil.rmtree(f"{path_data}state/")
# os.makedirs(f"{path_data}state/")

In [ ]:
import os
done = os.listdir(f"{path_data}resume_eval/")
done = [x.replace(".parquet","") for x in done]
falta = [x for x in symbs if x not in done]
print("Done symbols:", len(done), done)
print("Missings    :", len(falta), falta, "\n")

for symb in [x for x in symbs if x not in done]:

    print(symb)
    _, df = process_signals(symb, "1h", DAYS)
    
    df   = df[(df['date'] >= dates[0]) & (df['date'] <= dates[1])]

    cols_base = [x for x in df if x not in cols_strsig] 
    order = cols_base + sorted(cols_strsig)
    df = df[order]
    df[cols_strsig] = df[cols_strsig].replace({"": 0}).fillna(0).astype(int)
    df[cols_strsig] = df[cols_strsig].fillna(0).astype(int)
    

    def def_run_signal(df, name_signal):
        try:
            res, state = def_engine(df, name_signal)
        except Exception as e:
            print(e)
            print("ERROR: run_signal():", signal)
            res = pd.DataFrame()
            state = pd.DataFrame()
            
        return name_signal, res, state

    # 1. Parallel
    results = Parallel(n_jobs=-1, backend="loky")(delayed(def_run_signal)(df, name_signal) for name_signal in cols_strsig)
    resume = []
    state = []
    for _name_signal, _resume, _state in results:
        resume.append(_resume)
        state.append(_state)

    # 2. Secuencial
    # resume = []
    # state = []
    # for name_signal in cols_strsig[:3]:
    #     _, _resume, _state = run_signal(df, name_signal)
    #     resume.append(_resume)
    #     state.append(_state)

    resume = pd.concat(resume)
    state = pd.concat(state)

    resume["symbol"] = symb
    state["symbol"] = symb

    resume.to_parquet(f"{path_data}resume_eval/{symb}.parquet") # resume_predict
    import joblib
    joblib.dump(state, f"{path_data}state_eval/{symb}.pkl") # state_predict

    print("="*120)

In [ ]:
import os
p = path_data + "resume_eval/"
# p = path_data + "resume_predict/"
files = os.listdir(p)
resume = pd.concat([pd.read_parquet(p + file) for file in files]).sort_values("capital", ascending=False)
print(len(resume))
c = len(resume[resume["pnl"] < 0])
print("< 0:", c, f" | {(c/len(resume)*100):.1f}%")

c = len(resume[resume["pnl"] > 0])
print("> 0:", c, f" | {(c/len(resume)*100):.1f}%")

c = len(resume[resume["pnl"] > 10])
print("> 20:", c, f" | {(c/len(resume)*100):.1f}%")

print(f"PnL: ${resume["pnl"].sum():.1f}")

resume.head(10)

In [ ]:
# resume = resume[~resume["signal"].isin(exclude_signals)]
resume = resume[resume["capital"] > 1010]
resume

In [ ]:
resume["signal"].value_counts(normalize=True).head(15)*100

In [ ]:
print(len(resume))
resume.head(10)

In [ ]:
resume.tail(30)

In [ ]:
resume["symbol"].value_counts(normalize =True).head(20)*100

In [ ]:
resume["signal"].value_counts(normalize =True).head(20)*100

In [ ]:
ax = resume.groupby("signal").agg({"pnl":"sum", "trades":"sum", "return":"sum", "tradesxday":"sum", "mean_profit":"mean"}).reset_index().sort_values("pnl", ascending=False)
ax["porc"] = (ax["pnl"] / ax["pnl"].sum())*100

best_signals = list(set(resume["signal"].tolist()))
best_symbols = list(set(resume["symbol"].tolist()))
print(len(best_symbols), best_symbols)

print("\nPnL   :", ax["pnl"].sum())
print("Return:", ax["return"].sum())

ax

In [ ]:
ax2 = resume.groupby(["symbol", "signal"]).agg({"pnl":"sum", "trades":"sum", "return":"sum", "tradesxday":"sum", "mean_profit":"mean"}).reset_index().sort_values(["symbol", "pnl"], ascending=False)
ax2["count"] = 1
ax2["rank"] = ax2.groupby("symbol")["count"].transform("cumsum")
del ax2["count"]
ax2

In [ ]:
ax2.groupby(["signal"]).agg({"rank":"mean", "trades":"sum", "pnl":"sum"}).reset_index().sort_values(["rank"], ascending=True)


In [ ]:
ax2[ax2["rank"] <= 1]

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

dfplot = ax2[ax2["rank"] <= 1]
x_cat = "symbol"      
hue_cat = "signal"    
y_num = "pnl" 
aggfunc="mean"
figsize=(15, 5)

if aggfunc == "mean":
    orden = (
        dfplot.groupby(x_cat)[y_num]
        .mean()
        .sort_values()
        .index
    )
elif aggfunc == "sum":
    orden = (
        dfplot.groupby(x_cat)[y_num]
        .sum()
        .sort_values()
        .index
    )

plt.figure(figsize=figsize)

ax = sns.barplot(
    data=dfplot,
    x=x_cat,
    y=y_num,
    hue=hue_cat,
    order=orden,
    width = 0.8,
    edgecolor="white",   
    linewidth=2       
    # estimator=getattr(np, aggfunc),
    # errorbar=None
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=2,
        fontsize = 9.5 
    )
# plt.xticks()
plt.xlabel(y_num)
plt.ylabel(x_cat)
plt.title(f"{y_num} por {x_cat}")
plt.legend(fontsize = 8.5)
plt.tight_layout()
plt.show()

In [ ]:
dfplot["signal"].value_counts(normalize =True).head(20)*100

In [ ]:
revisar cuales guardar. revisar si sacar por rank o que metrica

In [ ]:
a = ax2[ax2["rank"] <= 1]
dicc = {}
for i in range(len(a)):
    dicc[a.iloc[i]["symbol"]] = a.iloc[i]["signal"]
import joblib
joblib.dump(dicc, path_repo + "pkl/best_signals_symbols.pkl")
dicc

In [ ]:
symbs = [x for x,y in dicc.items()]
print(symbs)

In [ ]:
import os, joblib
p = path_data + "state_eval/"
files = os.listdir(p)
state = pd.concat([joblib.load(p + file) for file in files])#.sort_values("capital", ascending=False)
print(len(state))
state = state.reset_index().rename(columns = {"level_0":"strat"}).drop(columns = ["level_1", "id"])

state.head(10)

In [ ]:
qwe

## 3. Test de una señal en varios simbolos

In [ ]:
name_signal = "str_macd_div1"

In [ ]:
from joblib import Parallel, delayed
def run_signal(symb):
    
    _, df = process_signals(symb, "1h", 500)

    cols_base = [x for x in df if x not in cols_strsig] 
    order = cols_base + sorted(cols_strsig)
    df = df[order]

    for signal in [name_signal]:
        try:
            mapping = {-1: -1, "": 0, 1: 1}
            df[signal] = df[signal].replace(mapping).fillna(0).astype(int)
        except:
            print("ERROR:", signal)
   
    df = df[(df['date'] >= FECHA_INICIO) & (df['date'] <= FECHA_FIN)]
    print(df.shape)
    print(df["date"].min() , "|" ,df["date"].max())
    res, state = def_engine(df, name_signal)
    res["symbol"] = symb
    state["symbol"] = symb
    df["symbol"] = symb
        
    return res, state, df

In [ ]:
results = Parallel(n_jobs=-1, backend="loky")(delayed(run_signal)(symb) for symb in symbs)
resume = []
state = []
df = []

for res, state, _df in results:
    resume.append(res)
    state.append(state)
    df.append(_df)

resume = pd.concat(resume).sort_values("capital", ascending = False)
state = pd.concat(state)
df = pd.concat(df)

resume.to_parquet(path_data + "3_onesignal_allsymbs/resume.parquet")
state.to_parquet(path_data + "3_onesignal_allsymbs/state.parquet")
df.to_parquet(path_data + "3_onesignal_allsymbs/df.parquet")

resume2 = resume.copy()
resume

In [ ]:
resume = pd.read_parquet(path_data + "3_onesignal_allsymbs/resume.parquet")
state = pd.read_parquet(path_data + "3_onesignal_allsymbs/state.parquet")
df = pd.read_parquet(path_data + "3_onesignal_allsymbs/df.parquet")

In [ ]:
best_symbols = resume[resume["pnl"] > 0]["symbol"].tolist()
print(len(best_symbols))
print(best_symbols)
import joblib
joblib.dump(best_symbols, path_data + "3_onesignal_allsymbs/best_symbols.pkl") cambiar ruta

In [ ]:
resume = resume[resume["pnl"] > 0]

In [ ]:
resume[['pnl', 'return', 'trades','tradesxday', 'total_gains', 'total_loss','close_tp', 'close_sl', 'close_ts']].sum()

In [ ]:
resume[['capital', 'win_rate', 'pnl', 'return', 'trades','tradesxday', 'drawdown', 'sharpe', 'ratio_wl', 'total_gains', 'total_loss', 'mean_timextrade', 'mean_profit', 'min_profit',
        'max_profit', 'mean_position', 'expectancy', 'recov_factor', 'max_rach_loss', 'max_rach_gain', 'close_tp', 'close_sl', 'close_ts']].describe().T.drop(columns = [
        "count", "std", "25%", "75%"])

In [ ]:
resume.head(10)

Trades per day

In [ ]:
state["ymd"] = state["time"].dt.strftime("%Y-%m-%d")
state["hour"] = state["time"].dt.strftime("%H").astype(int)
ax = state[(state["hour"] >= 8) & (state["hour"] <= 18)]
# ax = state
ax = ax.groupby(["symbol", "ymd"])["signal"].count().reset_index()
ax = pd.pivot_table(ax, index=["ymd"], columns="symbol", fill_value=0).reset_index()
ax.columns = ax.columns.get_level_values(level=1)
ax.rename(columns = {"":"time"}, inplace =True)
ax.tail(5)

In [ ]:
best_symbols = list(dict.fromkeys(best_symbols))
ax["total"] = ax[best_symbols].sum(axis=1)
ax2 = ax.groupby(["time"])["total"].sum().reset_index()#.tail(10)
ax2 = ax2[ax2["time"]>="2026-02-30"]
ax2.tail(10)

In [ ]:
# Days with more trades
ax.sort_values("total", ascending = False).head(5)

In [ ]:
# Trades per day
ax.describe().T.reset_index().sort_values("mean", ascending = False).head(10)

In [ ]:
fig, axi = plt.subplots(figsize=(16, 2.4))
color_back = "#0d1117"
fig.patch.set_facecolor(color_back)
axi.set_facecolor(color_back)
axi.plot(ax2["time"], ax2["total"], color="#38bdf8", linewidth=1.7, zorder=2, marker = "o", markersize = 3.5)
axi.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.xticks(ticks = ax2["time"], labels = ax2["time"].str[-5:], rotation=90, color="#94a3b8", fontsize=7.4)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in axi.spines.values():
    spine.set_edgecolor("#1e293b")
axi.grid([])
# axi.set_title("Equity", color="#e2e8f0", fontsize=10, fontweight="bold", pad=14)
# axi.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
axi.set_ylabel("", color="#64748b", fontsize=10, labelpad=8)
# axi.set_ylim(state["Close"].min()*0.993, state["Close"].max()*1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, axi = plt.subplots(figsize=(16, 3))
color_back = "#0d1117"
fig.patch.set_facecolor(color_back)
axi.set_facecolor(color_back)
ax3 = ax[ax["time"]>="2026-04-30"]
for symb in best_symbols[:7]:
    axi.plot(ax3["time"], ax3[symb], linewidth=1.7, zorder=2, label = symb, marker = "o", markersize = 0)

axi.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in axi.spines.values():
    spine.set_edgecolor("#1e293b")
axi.grid([])
# axi.set_title("Equity", color="#e2e8f0", fontsize=10, fontweight="bold", pad=14)
# axi.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
axi.set_ylabel("Equity", color="#64748b", fontsize=10, labelpad=8)
# axi.set_ylim(state["Close"].min()*0.993, state["Close"].max()*1.01)
plt.legend(loc = "upper left")
plt.tight_layout()
plt.show()


In [ ]:
ax = state.copy()#[(state["hour"] >= 8) & (state["hour"] <= 18)]
# ax = state
ax = ax.groupby(["symbol", "hour"])["signal"].count().reset_index()
ax = pd.pivot_table(ax, index=["hour"], columns="symbol", fill_value=0).reset_index()
ax.columns = ax.columns.get_level_values(level=1)
ax.rename(columns = {"":"hour"}, inplace =True)
ax["total"] = ax[best_symbols].sum(axis=1)
ax.head(4)

In [ ]:
ax2 = ax[["hour", "total"]]
fig, axi = plt.subplots(figsize=(16, 2.2))
fig.patch.set_facecolor(color_back)
axi.set_facecolor(color_back)
axi.bar(ax2["hour"], ax2["total"], color="#38bdf8", zorder=2)
# axi.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--", zorder=2)
for _, row in ax2.iterrows():
    axi.annotate(f"{row["total"]:.0f}", xy=(row["hour"], row["total"]), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8.5, color="#c1dbca", fontweight="bold")
plt.xticks(ax2["hour"], rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in axi.spines.values():
    spine.set_edgecolor("#1e293b")
axi.grid([])
axi.set_ylabel("", color="#64748b", fontsize=10, labelpad=8)
plt.ylim(0, ax2["total"].max()*1.2)
plt.tight_layout()
plt.show()


ax2["cat"] = np.where((ax2["hour"] >= 1) & (ax2["hour"] <= 4), "1-4", 
             np.where((ax2["hour"] >= 5) & (ax2["hour"] <= 8), "5-8", 
             np.where((ax2["hour"] >= 9) & (ax2["hour"] <= 12), "9-12", 
             np.where((ax2["hour"] >= 13) & (ax2["hour"] <= 16), "13-16", 
             np.where((ax2["hour"] >= 17) & (ax2["hour"] <= 20), "17-20", "21-0")))))
ax3 = ax2.groupby(["cat"]).total.sum().reset_index()
ax3["cat"] = pd.Categorical(ax3["cat"], ['1-4', "5-8", "9-12", '13-16', '17-20', '21-0'])
ax3 = ax3.sort_values("cat")
fig, axi = plt.subplots(figsize=(16, 2.2))
fig.patch.set_facecolor(color_back)
axi.set_facecolor(color_back)
axi.bar(ax3["cat"], ax3["total"], color="#38bdf8", zorder=2)
# axi.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--", zorder=2)
for _, row in ax3.iterrows():
    axi.annotate(f"{row["total"]:.0f}", xy=(row["cat"], row["total"]), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8.5, color="#c1dbca", fontweight="bold")
plt.xticks(ax3["cat"], rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in axi.spines.values():
    spine.set_edgecolor("#1e293b")
axi.grid([])
axi.set_ylabel("", color="#64748b", fontsize=10, labelpad=8)
plt.ylim(0, ax3["total"].max()*1.2)
plt.tight_layout()
plt.show()

In [ ]:
ax = state.copy()
ax = ax.groupby(["symbol", "ymd"])["pnl"].sum().reset_index() # return% profit pnl
ax = pd.pivot_table(ax, index=["ymd"], columns="symbol", fill_value=0).reset_index()
ax.columns = ax.columns.get_level_values(level=1)
ax.rename(columns = {"":"time"}, inplace =True)
ax["total"] = ax[best_symbols].sum(axis=1)

fig, axi = plt.subplots(figsize=(16, 3))
color_back = "#0d1117"
fig.patch.set_facecolor(color_back)
axi.set_facecolor(color_back)
axi.plot(ax["time"], ax["total"], color="#38bdf8", linewidth=1.7, zorder=2, marker = "o", markersize = 0)
axi.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.xticks(ticks = ax["time"], labels = ax["time"].str[-5:], rotation=90, color="#94a3b8", fontsize=2)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in axi.spines.values():
    spine.set_edgecolor("#1e293b")
axi.grid([])
# axi.set_title("Equity", color="#e2e8f0", fontsize=10, fontweight="bold", pad=14)
# axi.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
axi.set_ylabel("", color="#64748b", fontsize=10, labelpad=8)
# axi.set_ylim(state["Close"].min()*0.993, state["Close"].max()*1.01)
plt.tight_layout()
plt.show()


In [ ]:
ver donde guardar

In [ ]:
pd.set_option('display.max_rows', 100)
print(f"Capital      : ${resume["capital"].min():.0f} | ${resume["capital"].max():.0f} | ${resume["capital"].mean():.0f}")
print(f"Win rate     : {resume["win_rate"].min():.0f}% | {resume["win_rate"].max():.0f}% | {resume["win_rate"].mean():.0f}%")
print(f"PnL          : ${resume["pnl"].min():.0f} | ${resume["pnl"].max():.0f} | ${resume["pnl"].mean():.0f}")
print(f"Return       : {resume["return"].min():.0f}% | {resume["return"].max():.0f}% | {resume["return"].mean():.0f}%")
print(f"Trades       : {resume["trades"].min():.0f} | {resume["trades"].max():.0f} | {resume["trades"].mean():.0f}")
print(f"Trades x Day : {resume["tradesxday"].min():.1f} | {resume["tradesxday"].max():.1f} | {resume["tradesxday"].mean():.1f}")
print(f"Max rach gain: {resume["max_rach_gain"].min():.0f} | {resume["max_rach_gain"].max():.0f} | {resume["max_rach_gain"].mean():.0f}")
print(f"Max rach loss: {resume["max_rach_loss"].min():.0f} | {resume["max_rach_loss"].max():.0f} | {resume["max_rach_loss"].mean():.0f}")
resume["atr_sl"] = atr_sl
resume["multiplier_tp"] = multiplier_tp
resume["trailing_atr_mult"] = trailing_atr_mult
resume["use_trailing_stop"] = use_trailing_stop
resume["trailing_activation_r"] = trailing_activation_r
resume["breakeven_trigger_r"] = breakeven_trigger_r
if save:
    resume.to_parquet(f"results/{symb}.parquet")
    state.to_parquet(f"df_symbol/{symb}.parquet")

In [ ]:
resume.head(15)

In [ ]:
print(resume[resume["pnl"] > 2]["signal"].tolist())

In [ ]:
state2 = state["str_macd_div1"]  # str_macd_div1
state2["EMA"] = state2["close"].rolling(window = 200).mean()
state2["EMA2"] = state2["close"].rolling(window = 50).mean()
state2["std"] = state2["close"].rolling(20).std()
state2["bb_low"] = state2["close"] - 2*state2["std"]
state2["bb_upper"] = state2["close"] + 2*state2["std"]
state2["close_exit"] = np.where((state2["state"] == "EXIT") & (state2["reason"] == "TP"), state2["tp"].shift(1), 
                      np.where((state2["state"] == "EXIT") & (state2["reason"] == "SL"), state2["sl"].shift(1), state2["close"]))

print(state2.shape)
# state = state2.copy().iloc[8500:11000,]
state = state2#[state2["time"] >= "2026-05-10"]
print(state["time"].min(), "|", state["time"].max())
print(state.shape)

In [ ]:
state2

In [ ]:
state.head(5)

In [ ]:
# Price
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
titlesize = 10
edgecolor = "#A3B0C4"
colorlines = "#398bae"

state.index = state["time"]
state["std"] = state["close"].rolling(20).std()
state["bb_low"] = state["close"] - 2*state["std"]
state["bb_upper"] = state["close"] + 2*state["std"]
state["MACD_hist"] = df["MACD_hist"]
state["MACD_signal"] = df["MACD_signal"]
state["MACD"] = df["MACD"]

state["time"] = pd.to_datetime(state["time"])
state["Close"] = state["close"]
sell = state[state["signal"] == -1]
buy  = state[state["signal"] ==  1]
exits = state.copy()
exits = exits[exits["state"] == "EXIT"]

# fig, ax = plt.subplots(figsize=(16, 5))
fig, (ax1, ax5, ax2, ax3, ax4) = plt.subplots(
    5, 1,
    figsize=(15, 7.4),
    sharex=True,
    gridspec_kw={'height_ratios': [1.8, 0.35, 0.35, 0.22, 0.2]}  
)
color_back = "#0d1117"
fig.patch.set_facecolor(color_back)

ax1.set_facecolor(color_back)
ax1.plot(state["time"], state["Close"], color="#ffffff", linewidth=1.1, zorder=2, alpha = 0.8)
# ax1.plot(state["time"], state["high"], color="#184157", linewidth=1.1, zorder=2)
# ax1.plot(state["time"], state["low"], color="#18445c", linewidth=1.1, zorder=2)

# # EMAS
# ax1.plot(state["time"], state["EMA"], color="#610871", linewidth=1, zorder=2, linestyle = "-", label = "EMA200")
# ax1.plot(state["time"], state["EMA2"], color="#6F4C0C", linewidth=1, zorder=2, linestyle = "-", label = "EMA50")
# # Bollinger
# ax1.plot(state["time"], state["bb_low"], color="#143868", linewidth=1, zorder=2, linestyle = "--")
# ax1.plot(state["time"], state["bb_upper"], color="#143868", linewidth=1, zorder=2, linestyle = "--")

# SL
if "sl" in state.columns:
    sl_data = state
    ax1.plot(sl_data["time"], sl_data["sl"], color="#ef4444", linewidth=1.1, alpha = 0.8, linestyle="-", zorder=3)#, marker = "o", markersize = 1)

# TP
if "tp" in state.columns:
    tp_data = state
    ax1.plot(tp_data["time"], tp_data["tp"], color="#44ef52", linewidth=1.1, alpha = 0.8, linestyle="-", zorder=3)

# Sell signals
if not sell.empty:
    # ax1.scatter(sell["time"], sell["Close"], color="#f87171", s=45, zorder=5, edgecolors="white", linewidths=1.1)
    for _, row in sell.iterrows():
        ax1.annotate("▼", xy=(row["time"], row["Close"]), xytext=(0, 0),  zorder=6, textcoords="offset points", ha="center", fontsize=11.5, color="#f87171", fontweight="bold")

# Buy signals
if not buy.empty:
    # ax1.scatter(buy["time"], buy["Close"], color="#4ade80", s=45, zorder=5, edgecolors="white", linewidths=1.1)
    for _, row in buy.iterrows():
        ax1.annotate("▲", xy=(row["time"], row["Close"]), xytext=(0, 0),  zorder=6, textcoords="offset points", ha="center", fontsize=11.5, color="#4ade80", fontweight="bold")

if not exits.empty:
    for _, row in exits.iterrows():
        if row["reason"]  == "SL":
            _reas = "SL"
        elif row["reason"] == "TP":
            _reas = "TP"
        elif row["reason"] == "TS": 
            _reas = "TS"
        elif row["reason"] == "REV":
            _reas = "RE"

        if row["return%"] > 0:
            color = "#43d75f"
        if row["return%"] < 0:
            color = "#e96161"

        # ax1.annotate(t, xy=(row["time"], row["Close"]*0.995), xytext=(0, 30), textcoords="offset points", ha="center", fontsize=11, color=color, fontweight="bold")
        ax1.annotate(f"{row["return%"]*100:.0f}%", xy=(row["time"], row["close_exit"]*0.995), xytext=(-5, 30), textcoords="offset points", ha="center", fontsize=8.8, 
                    color=color, fontweight="bold")
        
        ax1.annotate(_reas, xy=(row["time"], row["close_exit"]*0.995), xytext=(-5, 40), textcoords="offset points", ha="center", fontsize=7.1, 
                    color="#94a3b8", fontweight="bold")
        
    ax1.scatter(exits["time"], exits["close_exit"], color="#D3C90F", s=38, zorder=5, marker = "D", edgecolors=color_back, linewidths=1.2, label="")
# ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax1.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax1.spines.values():
    spine.set_edgecolor(edgecolor)

ax1.grid(False)
ax1.set_title("", color="#e2e8f0", fontsize=13, fontweight="bold", pad=14)
# ax1.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax1.set_ylabel("Precio (USDT)", color="#64748b", fontsize=10, labelpad=8)
_min = min(state["sl"].min(), state["tp"].min(), state["Close"].min())*0.989
_max1 = max(state["sl"].max(), state["tp"].max(), state["Close"].max())*1.011
ax1.set_ylim(_min, _max1)
# plt.legend(loc = "upper right", fontsize = 7, framealpha = 0.3, labelcolor = "white", frameon = True, handlelength=3.5, handleheight=1)


# MACD
# fig, ax5 = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax5.set_facecolor(color_back)

colors = ["#22c55e" if x >= 0 else "#ef4444" for x in state["MACD_hist"]]
ax5.bar(state["time"], state["MACD_hist"]*1.95, color=colors, width=0.05, alpha=1, zorder=1, label="Histograma")

ax5.plot(state["time"], state["MACD"], color=colorlines, linewidth=1.5, zorder=3, label="MACD")
# ax5.plot(state["time"], state["MACD_signal"], color="#b79030", linewidth=1.2, zorder=4, label="Signal")

ax5.axhline(0, color="#FFFFFF", linewidth=1, linestyle="--", zorder=2, alpha = 0.5) 
ax5.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax5.spines.values():
    spine.set_edgecolor(edgecolor)
ax5.grid(False) 
ax5.set_ylabel("MACD", color="#64748b", fontsize=10)


# ADX
profitcum = state
# fig, ax2 = plt.subplots(figsize=(16, 1.5))
fig.patch.set_facecolor(color_back)
ax2.set_facecolor(color_back)
ax2.plot(profitcum["time"], profitcum["ADX"], color=colorlines, linewidth=1,  zorder=2)
ax2.plot(profitcum["time"], profitcum["ADX+DI"], color="#4ade80", linewidth=1,  zorder=2)
ax2.plot(profitcum["time"], profitcum["ADX-DI"], color="#ff7041", linewidth=1,  zorder=2)
ax2.axhline(20, color="#ffffff", linewidth=1, linestyle="--",zorder=2, alpha = 0.5)
ax2.axhline(40, color="#ffffff", linewidth=1, linestyle="--",zorder=2, alpha = 0.5)
# ax2.xax2is.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax2.xax2is.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax2.spines.values():
    spine.set_edgecolor(edgecolor)

ax2.grid([])
# ax2.set_title("ADX", color="#e2e8f0", fontsize=10, fontweight="bold", pad=14)
# ax2.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax2.set_ylabel("ADX", color="#64748b", fontsize=10, labelpad=8)
ax2.set_ylim(0, 70)


# RSI
profitcum = state
# fig, ax3 = plt.subplots(figsize=(16, 1.5))
fig.patch.set_facecolor(color_back)
ax3.set_facecolor(color_back)
ax3.plot(profitcum["time"], profitcum["RSI"], color=colorlines, linewidth=1,  zorder=2)
ax3.axhline(30, color="#ffffff", linewidth=1, linestyle="--",zorder=2, alpha = 0.5)
ax3.axhline(70, color="#ffffff", linewidth=1, linestyle="--",zorder=2, alpha = 0.5)
# ax3.xax3is.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax3.xaxis.set_major_locator(mdates.HourLocator(interval=1))

plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax3.spines.values():
    spine.set_edgecolor(edgecolor)
ax3.grid([])
# ax3.set_title("RSI", color="#e2e8f0", fontsize=10, fontweight="bold", pad=14)
# ax3.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax3.set_ylabel("RSI", color="#64748b", fontsize=10, labelpad=8)
ax3.set_ylim(0, 100)


# Slope
profitcum = state.copy()
profitcum["profitcum"] = profitcum["close"].rolling(20).mean()
profitcum["profitcum"] = profitcum["profitcum"].diff()
# profitcum["returncum"] = profitcum["return%"].cumsum()
# fig, ax4 = plt.subplots(figsize=(16, 2.2))
fig.patch.set_facecolor(color_back)
ax4.set_facecolor(color_back)
ax4.plot(profitcum["time"], profitcum["profitcum"], color=colorlines, linewidth=1,zorder=2)

ax4.axhline(0, color="#ffffff", linewidth=1, linestyle="--",zorder=2, alpha = 0.5)
# ax4.xax4is.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax4.xax4is.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax4.spines.values():
    spine.set_edgecolor(edgecolor)
ax4.grid([])
# ax4.set_title("Profit Acum $", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax4.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax4.set_ylabel("Slope $", color="#64748b", fontsize=10, labelpad=8)
# ax4.set_ylim(_min, _max4)

for ax in [ax1, ax2, ax3]:
    ax.tick_params(axis='y', colors="#94a3b8")
    # ax.tick_params(axis='x', colors="#94a3b8")
    ax.tick_params(axis='both', labelsize=8)

plt.tight_layout()
plt.show()



In [ ]:
heightplots = 1.9
# Equity
state["Close"] = state["patrimony"]
sell = state[state["signal"] == -1]
buy  = state[state["signal"] ==  1]
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
ax.plot(state["time"], state["Close"], color="#38bdf8", linewidth=1.7, zorder=2, label="Close Price")
ax.axhline(capital, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Equity", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Equity", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylim(state["Close"].min()*0.993, state["Close"].max()*1.01)
plt.tight_layout()
plt.show()



# Profit Acum
exits = state.copy()
profitcum = exits[exits["state"] == "EXIT"]
profitcum = profitcum[["time", "profit", "return%", "reason"]]
profitcum["profitcum"] = profitcum["profit"].cumsum()
# profitcum["returncum"] = profitcum["return%"].cumsum()
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
ax.plot(profitcum["time"], profitcum["profitcum"], color="#38bdf8", linewidth=1.4, marker = "o", markersize = 2.5, zorder=2)

nfactor = 2
lista = [x for x in range(len(profitcum)) if x % nfactor == 0]

for _, row in profitcum.iloc[lista].iterrows():
    if row["profitcum"] > 0: # row["reason"]  == "TP":
        color = "#4ade80"
        ypos = 8
    else:
        color = "#ff7041"
        ypos = -11
    ax.annotate(f"{row["profitcum"]:.0f}", xy=(row["time"], row["profitcum"]), xytext=(0, ypos), textcoords="offset points", ha="center", fontsize=7, color=color, fontweight="bold")
ax.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.annotate(f"${profitcum["profitcum"].iloc[-1]:.1f}", xy=(1, 1), xycoords="axes fraction", xytext=(-0, 16), textcoords="offset points", ha="right", va="top", fontsize = 11,
             weight = "bold", color = "#c5d6f1")
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Profit Acum $", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Profit $", color="#64748b", fontsize=10, labelpad=8)
_min = profitcum["profitcum"].min()*1.8
_max = profitcum["profitcum"].max()*1.35
# ax.set_ylim(_min, _max)
plt.tight_layout()
plt.show()



# Returns acum
# profitcum = exits[["time", "profit", "return%", "reason"]]
profitcum["return%"] = profitcum["return%"]*100
profitcum["profitcum"] = profitcum["return%"].cumsum()
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
ax.plot(profitcum["time"], profitcum["profitcum"], color="#38bdf8", linewidth=1.4, marker = "o", markersize = 2.5, zorder=2)

lista = [x for x in range(len(profitcum)) if x % nfactor == 0]

for _, row in profitcum.iloc[lista].iterrows():
    if row["profitcum"] > 0: # row["reason"]  == "TP":
        color = "#4ade80"
        ypos = 8
    else:
        color = "#ff7041"
        ypos = -11
    ax.annotate(f"{row["profitcum"]:.0f}", xy=(row["time"], row["profitcum"]), xytext=(0, ypos), textcoords="offset points", ha="center", fontsize=7, color=color, fontweight="bold")
ax.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
plt.annotate(f"{profitcum["profitcum"].iloc[-1]:.1f}%", xy=(1, 1), xycoords="axes fraction", xytext=(-0, 16), textcoords="offset points", ha="right", va="top", fontsize = 11,
             weight = "bold", color = "#c5d6f1")
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Return acum %", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Return %", color="#64748b", fontsize=10, labelpad=8)
_min = profitcum["profitcum"].min()*1.8
_max = profitcum["profitcum"].max()*1.35
# ax.set_ylim(_min, _max)
plt.tight_layout()
plt.show()


# Returns
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
exits = state[state["state"] == "EXIT"]
exits["return%"] = exits["return%"]*100
ax.plot(exits["time"], exits["return%"], color="#38bdf8", linewidth=1.4, marker = "o", markersize = 2.5, zorder=2)

lista = [x for x in range(len(exits)) if x % nfactor == 0]

for _, row in exits.iloc[lista].iterrows():
    if row["reason"]  == "TP":
        color = "#4ade80"
        ypos = 8
    if row["reason"]  == "SL":
        color = "#ff7041"
        ypos = -11
    ax.annotate(f"{row["return%"]:.1f}", xy=(row["time"], row["return%"]), xytext=(0, ypos), textcoords="offset points", ha="center", fontsize=7, color=color, fontweight="bold")
ax.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Return %", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Return %", color="#64748b", fontsize=10, labelpad=8)
_min = exits["return%"].min()*1.8
_max = exits["return%"].max()*1.35
# ax.set_ylim(_min, _max)
plt.tight_layout()
plt.show()



# Entry amount
exits = state.copy()
exits["entry_amount"] = exits["entry_amount"].shift(1)
profitcum = exits[exits["state"] == "EXIT"]
profitcum["profitcum"] = profitcum["entry_amount"]
# profitcum["returncum"] = profitcum["return%"].cumsum()
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
ax.plot(profitcum["time"], profitcum["profitcum"], color="#38bdf8", linewidth=1.4, marker = "o", markersize = 2.5, zorder=2)

lista = [x for x in range(len(profitcum)) if x % nfactor == 0]

for _, row in profitcum.iloc[lista].iterrows():
    ax.annotate(f"{row["profitcum"]:.1f}", xy=(row["time"], row["profitcum"]), xytext=(0, 8), textcoords="offset points", ha="center", fontsize=7, color="#4ade80", fontweight="bold")
# ax.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
# plt.annotate(f"${profitcum["profitcum"].iloc[-1]:.1f}", xy=(1, 1), xycoords="axes fraction", xytext=(0, 16), textcoords="offset points", ha="right", va="top", fontsize = 11,
#              weight = "bold", color = "#c5d6f1")
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
prom = profitcum["profitcum"].mean()
plt.annotate(f"prom: ${prom:.0f}", xy=(1, 1), xycoords="axes fraction", xytext=(-0, 16), textcoords="offset points", ha="right", va="top", fontsize = 10, weight = "bold", 
             color = "#c5d6f1")
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Entry amount $", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Amount $", color="#64748b", fontsize=10, labelpad=8)
_min = profitcum["profitcum"].min()*1.8
_max = profitcum["profitcum"].max()*1.35
# ax.set_ylim(_min, _max)
plt.tight_layout()
plt.show()



# Drawdown
fig, ax = plt.subplots(figsize=(16, heightplots))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
drawd = state.copy()
drawd["drawdown"] = drawd["drawdown"]*100
ax.plot(drawd["time"], drawd["drawdown"], color="#38bdf8", linewidth=1.1, zorder=2, label="Close Price")
ax.axhline(0, color="#ffffff", linewidth=1.4, linestyle="--",zorder=2)
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
# ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
ax.set_title("Drawdown %", color="#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
# ax.set_xlabel("Hora (UTC)", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylabel("Drawdown %", color="#64748b", fontsize=10, labelpad=8)
ax.set_ylim(drawd["drawdown"].min()*1.2, max(drawd["drawdown"].max()*1.1, 0.2))
plt.tight_layout()
plt.show()


# Hist returns
fig, ax = plt.subplots(figsize=(6.5, 3.4))
fig.patch.set_facecolor(color_back)
ax.set_facecolor(color_back)
exits = state[state["state"] == "EXIT"]
returns = (exits["return%"]*100)
plt.hist(returns, color="#38bdf8", density=True, bins = 25, edgecolor = color_back, linewidth = 4)
plt.xticks(rotation=0, color="#94a3b8", fontsize=9)
plt.yticks(color="#94a3b8", fontsize=9)
plt.axvline(np.mean(returns), linestyle = "--", color = "white")
plt.annotate(f"{np.mean(returns):.1f}%", xy=(1, 1), xycoords="axes fraction", xytext=(-0, 16), textcoords="offset points", ha="right", va="top", fontsize = 10, weight = "bold", 
             color = "#c5d6f1")
for spine in ax.spines.values():
    spine.set_edgecolor("#1e293b")
ax.grid([])
plt.title("Returns %", color = "#e2e8f0", fontsize=titlesize, fontweight="bold", pad=14)
plt.tight_layout()
plt.show()

In [ ]:
# Señales por dia
print(state["signal"].value_counts())
state["ymd"] = state["time"].dt.strftime("%Y-%m-%d")
state["hour"] = state["time"].dt.strftime("%H").astype(int)

state.groupby(["ymd"])["signal"].count().tail(25)

In [ ]:
state[(state["hour"] >= 8) & (state["hour"] <= 18)].groupby(["ymd"])["signal"].count().tail(25)

In [ ]:
qwe

## Results

In [ ]:
import os
p = os.listdir(os.getcwd() + "/results")
resall = []
resall_full = []
for i in p:
    print(i)
    ax = pd.read_parquet("results/" + i)
    # ax = ax[ax["capital"] >= 1100]
    resall.append(ax.head(5))
    resall_full.append(ax)
resall = pd.concat(resall).sort_values(["symbol", "capital"], ascending = False)
resall_full = pd.concat(resall_full).sort_values(["symbol", "capital"], ascending = False)
print(len(resall))
resall.head(20)

In [ ]:
ax = resall.drop_duplicates("symbol")
ax

In [ ]:
resall["signal"].value_counts(normalize = True)*100

In [ ]:
resall_full.groupby("signal").agg({"win_rate":"mean"}).reset_index().sort_values("win_rate", ascending = False)

In [ ]:
qwe

In [ ]:
"""
* Rentabilidad general
Capital inicial $1,000
El dinero con el que empezó la simulación. Base de comparación para todo lo demás.
Neutral

Capital final $1,119
Lo que quedó al terminar el período. Incluye todas las ganancias, pérdidas y comisiones.
Bueno

Retorno estrategia +11.95%
Ganaste $119 sobre $1,000 iniciales. Lo más importante: superaste ampliamente al Buy & Hold en el mismo período.
Bueno

Retorno Buy & Hold -29.43%
Si hubieras comprado y sostenido sin operar, habrías perdido el 29%. Tu estrategia le ganó al mercado por +41 puntos porcentuales. Esto es alpha real.
+41pp alfa


* Actividad de trading
Total trades 361
Operaciones ejecutadas en 60 días ≈ 6 trades por día. Es alto; puede indicar overtrading o que la estrategia es muy activa. Más trades = más comisiones acumuladas.
Alto

Ganadoras 41.3%
Solo 4 de cada 10 trades terminan en ganancia. Parece bajo, pero es viable si las ganancias son mucho mayores que las pérdidas (ver Ratio R/R).
Bajo

Racha perdedora máx 11
Hubo un momento con 11 trades consecutivos perdedores. Con 1% de riesgo por trade esto representa ~10% de pérdida acumulada. Psicológicamente muy difícil de sostener en vivo.
Vigilar

Racha ganadora máx 5
Lo mejor fue 5 trades ganadores seguidos. La asimetría (11 pérdidas vs 5 ganancias seguidas) es normal cuando el winrate es bajo.
Normal


* Calidad del sistema
Profit Factor 1.06
Por cada $1 perdido, ganas $1.06. Significa que el sistema es rentable pero muy al límite. Por encima de 1.5 se considera robusto; 1.06 es frágil ante cambios de 
mercado o aumento de comisiones.
Frágil

Ratio R/R promedio 1.59
En promedio, cuando ganas obtienes 1.59x lo que pierdes cuando pierdes. Esto explica por qué el sistema sobrevive con solo 41% de aciertos: las ganancias compensan las pérdidas.
Bueno

Max Drawdown -15.05%
La caída máxima desde un pico hasta un valle fue del 15%. Sobre $1,000 son $150 de pérdida temporal. Es aceptable; por debajo de -20% empieza a ser preocupante.
Aceptable


* Métricas de riesgo ajustado
Sharpe Ratio 1.00
Mide retorno por unidad de riesgo total. Igual a 1.0 significa que el retorno compensa exactamente la volatilidad. <1 = malo, 1-2 = aceptable, >2 = bueno, >3 = excelente.
Aceptable

Sortino Ratio 1.67
Como el Sharpe pero solo penaliza la volatilidad bajista (las caídas). Que sea mayor que el Sharpe (1.67 vs 1.00) significa que la volatilidad de la estrategia viene más 
de subidas que de bajadas. Buena señal.
Bueno

Calmar Ratio 0.79
Retorno anualizado dividido entre el drawdown máximo. 0.79 significa que el retorno no llega a compensar el peor momento de caída. Por encima de 1.0 es lo deseable.
Mejorable


* Riesgo por trade 1%
En cada entrada se arriesga el 1% del capital disponible. Con 361 trades y rachas perdedoras de 11, esto es lo que evitó una pérdida catastrófica.
Conservador

SL × 1.5 ATR: 1.5x
El Stop Loss se coloca a 1.5 veces el ATR del precio de entrada. Suficientemente lejos para no ser alcanzado por ruido normal, pero con la racha de 11 perdedores 
seguidos podría estar muy ajustado.
Normal

TP × 3.0 ATR: 3.0x
Take Profit a 3x ATR = ratio riesgo/beneficio teórico de 1:2. Combinado con el 41% de aciertos, la matemática cierra: 0.41×2 - 0.59×1 = 0.23 de esperanza matemática positiva por trade.
Bueno
"""

## Correlations of indicators

In [ ]:
signal_map = {
    1: 1,
    -1: -1,
    "": 0,
    "⚪":0,
}


In [ ]:
ax = corr.reset_index()
ax = pd.melt(ax, id_vars="index")
ax["value"] = abs(ax["value"])
ax = ax.sort_values("value", ascending = False)
ax = ax[ax["index"] != ax["variable"]].drop_duplicates("value")
ax.head(10)

## Portafolio y frontera eficiente

## Utils

In [ ]:
def clean_signal(df):
    # print("Signals:", df[df["signal"] != 0].shape[0])
    # display(df["signal"].value_counts(normalize = True, dropna = False))
    # display(df["signal"].value_counts(normalize = False, dropna = False))
    # print("-"*60)

    # Filtro de volatilidad
    ## Evita mercados muertos o demasiado caóticos
    df["vol"] = df["ATR"] / df["Close"]
    df["signal"] = np.where((df["vol"] < 0.002) | (df["vol"] > 0.08), 0, df["signal"]) 
    # print("Volatilidad:    ", df[df["signal"] != 0].shape[0])

    # Filtro RSI
    df["signal"] = np.where((df["signal"] == 1) & (df["RSI"] > 65), 0, 
                   np.where((df["signal"] == -1) & (df["RSI"] < 35), 0, df["signal"]))
    # print("RSI:            ", df[df["signal"] != 0].shape[0])

    # # Filtro contra tendencia
    # df["signal"] = np.where((df["signal"] == 1) & (df["trend"].isin(["▼ 2", "▼ 3", "▼ 4 XTREM", "▼ 5 SUPER XTREM"])), 0,  
    #                np.where((df["signal"] == -1) & (df["trend"].isin(["▲ 2", "▲ 3", "▲ 4 XTREM", "▲ 5 SUPER XTREM"])), 0, df["signal"]))
    # print("Contratendencia:", df[df["signal"] != 0].shape[0])

    # Filtro tendencia lateral
    df["signal"] = np.where((df["signal"] != 0) & (df["ADX"] < 20), 0, df["signal"])
    # print("ADX < 20:        ", df[df["signal"] != 0].shape[0])

    # display(df["signal"].value_counts(normalize = True, dropna = False))
    # display(df["signal"].value_counts(normalize = False, dropna = False))

    return df["signal"]